# Corporate Financial Analysis Tool

Author:zhixuan.Guo24
Date: April 2026
Course:ACC102  - Track 2

## Project Overview

This tool performs automated financial analysis on company financial statements, including:

- Data cleaning
- Feature engineering
- Visualization
- Multi-company comparison
- Financial health scoring
- Complete HTML report generation

Run all cells below in order and follow the prompts to enter your data source and configuration.
## Supported Data Sources

This tool supports two types of data sources.

1. Kaggle - Users can upload local CSV or Excel files downloaded from Kaggle. No login is required. Example: Sample Superstore dataset.

2. WRDS - Users can directly connect to the WRDS database to download Compustat (financial statements) or CRSP (stock prices and returns) data. WRDS login credentials are required.

Data Requirement: This tool is designed for corporate financial statement analysis. The dataset must contain revenue/sales and profit/net income columns. Non-financial data (e.g., credit scores, loan assessments, demographics) are not supported.

## 1. Import Libraries

Import all required Python packages:

- `wrds`: Connect to WRDS database
- `pandas` / `numpy`: Data manipulation
- `matplotlib` / `seaborn`: Visualization
- `getpass`: Secure password input

In [ ]:
# Install wrds package if needed (run once in terminal)
# !pip install wrds

import wrds
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import getpass
import warnings
warnings.filterwarnings('ignore')

# Set chart style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Set2")
plt.rcParams['font.sans-serif'] = ['Arial']

print("✅ All libraries imported successfully")


## 2. Data Source Selection

User chooses data source type:

- 1 - Kaggle: Upload local files (no login required)
- 2 - WRDS: Connect to Compustat/CRSP (WRDS login required)

Run this cell and enter 1 or 2 when prompted.

In [ ]:
# ============================================
# DATA SOURCE SELECTION
# ============================================

print("\n" + "=" * 80)
print("                    DATA SOURCE SELECTION")
print("=" * 80)

print("""
╔═══════════════════════════════════════════════════════════════════════╗
║                                                                       ║
║   [1]  KAGGLE - Any dataset from Kaggle                              ║
║        → Upload CSV/Excel file to current folder                     ║
║        → No login required                                           ║
║                                                                       ║
║   [2]  WRDS - Compustat (Financial Statement Data)                   ║
║        → Real public company financial data                          ║
║        → WRDS login required                                         ║
║        → Single or multi-company comparison                          ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
""")

while True:
    choice = input("👉 Enter 1 or 2: ")
    if choice in ['1', '2']:
        break
    print("❌ Please enter 1 or 2")

DATA_SOURCE_CHOICE = int(choice)

print("\n" + "=" * 80)
if DATA_SOURCE_CHOICE == 1:
    print("✅ Selected: KAGGLE mode")
    print("📁 Place your data file in the current folder")
else:
    print("✅ Selected: WRDS mode")
    print("🔐 You will be asked for WRDS username and password later")
print("=" * 80)
print("\n👉 NEXT: Run Cell 2 to configure parameters")


## 3. Configuration Setup

Based on the data source selection, user enters:

**Kaggle mode**:
- File name (e.g., SampleSuperstore.xlsx)
- Company/dataset name (for report title)

**WRDS mode**:
- Database type (financial or stock)
- Single or multiple companies (single or multiple)
- Stock tickers (e.g., AAPL,MSFT,GOOGL)
- Start year and end year

In [ ]:
# ============================================
# CONFIGURATION SETUP
# ============================================

print("\n" + "=" * 80)
print("                    CONFIGURATION SETUP")
print("=" * 80)

# ========== KAGGLE MODE ==========
if DATA_SOURCE_CHOICE == 1:
    DATA_SOURCE = 'kaggle'
    
    print("\n📁 KAGGLE MODE SELECTED")
    print("-" * 60)
    print("Please place your CSV or Excel file in the same folder as this notebook.")
    print("Supported formats: .xlsx or .csv")
    print("-" * 60)
    
    print("\nQUESTION 1: What is the name of your data file?")
    print("   Example: SampleSuperstore.xlsx")
    
    while True:
        KAGGLE_FILE = input("\n👉 File name: ")
        if KAGGLE_FILE.endswith('.xlsx') or KAGGLE_FILE.endswith('.csv'):
            break
        print("   ❌ File must end with .xlsx or .csv")
    
    print("\nQUESTION 2: What is the name of this company/dataset?")
    print("   This will appear as the report title.")
    
    COMPANY_NAME = input("\n👉 Company name: ") or "Kaggle Dataset"
    
    print("\n" + "-" * 60)
    print("✅ KAGGLE CONFIGURATION COMPLETE")
    print(f"   File: {KAGGLE_FILE}")
    print(f"   Title: {COMPANY_NAME}")
    print("-" * 60)

# ========== WRDS MODE ==========
elif DATA_SOURCE_CHOICE == 2:
    DATA_SOURCE = 'wrds'
    
    print("\n🔐 WRDS MODE SELECTED")
    print("-" * 60)
    print("You will need your WRDS username and password in Cell 3.")
    print("-" * 60)
    
    # ========== QUESTION 1: Choose database ==========
    print("\nQUESTION 1: Which WRDS database do you want to use?")
    print("   Type 'financial' for Compustat (financial statements)")
    print("   Type 'stock' for CRSP (stock prices and returns)")
    
    while True:
        db_choice = input("\n👉 Type 'financial' or 'stock': ").lower()
        if db_choice == 'financial':
            WRDS_DATABASE = 'compustat'
            print("   ✅ Selected: Compustat (Financial Statements)")
            break
        elif db_choice == 'stock':
            WRDS_DATABASE = 'crsp'
            print("   ✅ Selected: CRSP (Stock Prices and Returns)")
            break
        else:
            print("   ❌ Please type 'financial' or 'stock'")
    
    # ========== QUESTION 2: Single or Multiple ==========
    print("\nQUESTION 2: One company or multiple companies?")
    print("   Type 'single' or 'multiple'")
    
    while True:
        answer = input("\n👉 Type 'single' or 'multiple': ").lower()
        if answer == 'single':
            analysis_type = 'single'
            print("   ✅ Single company analysis")
            break
        elif answer == 'multiple':
            analysis_type = 'multiple'
            print("   ✅ Multi-company comparison")
            break
        else:
            print("   ❌ Please type 'single' or 'multiple'")
    
    # ========== QUESTION 3: Enter tickers ==========
    print("\nQUESTION 3: Enter stock ticker(s)")
    
    if WRDS_DATABASE == 'compustat':
        print("   Examples: AAPL (Apple), MSFT (Microsoft), GOOGL (Google)")
    else:
        print("   Examples: AAPL, MSFT, GOOGL, AMZN, TSLA")
    
    if analysis_type == 'single':
        print("   Enter ONE ticker (e.g., AAPL):")
    else:
        print("   Enter multiple tickers separated by commas (e.g., AAPL,MSFT,GOOGL):")
    
    while True:
        ticker_input = input("\n👉 Ticker(s): ")
        if ticker_input.strip():
            WRDS_COMPANIES = [t.strip().upper() for t in ticker_input.split(',')]
            print(f"   You entered: {', '.join(WRDS_COMPANIES)}")
            
            # Fix common typo: APPL -> AAPL
            if 'APPL' in WRDS_COMPANIES:
                WRDS_COMPANIES = ['AAPL' if x == 'APPL' else x for x in WRDS_COMPANIES]
                print(f"   ✅ Corrected to: {', '.join(WRDS_COMPANIES)}")
            break
        else:
            print("   ❌ Please enter at least one ticker")
    
    # ========== QUESTION 4: Start Year ==========
    print("\nQUESTION 4: What is the START year?")
    print("   Example: 2020")
    
    if WRDS_DATABASE == 'compustat':
        print("   Note: Compustat data available from 1960 onwards")
    else:
        print("   Note: CRSP data available from 1962 onwards")
    
    while True:
        try:
            WRDS_START_YEAR = int(input("\n👉 Start year: "))
            min_year = 1960 if WRDS_DATABASE == 'compustat' else 1962
            if WRDS_START_YEAR >= min_year:
                break
            print(f"   ❌ Year must be {min_year} or later")
        except:
            print("   ❌ Please enter a valid year (e.g., 2020)")
    
    # ========== QUESTION 5: End Year ==========
    print("\nQUESTION 5: What is the END year?")
    print(f"   Must be after {WRDS_START_YEAR}")
    print("   Example: 2024")
    
    while True:
        try:
            WRDS_END_YEAR = int(input("\n👉 End year: "))
            if WRDS_END_YEAR > WRDS_START_YEAR:
                break
            print(f"   ❌ End year must be greater than {WRDS_START_YEAR}")
        except:
            print("   ❌ Please enter a valid year (e.g., 2024)")
    
    # Build report title
    if len(WRDS_COMPANIES) == 1:
        COMPANY_NAME = f"WRDS/{WRDS_DATABASE.upper()}: {WRDS_COMPANIES[0]} ({WRDS_START_YEAR}-{WRDS_END_YEAR})"
    else:
        COMPANY_NAME = f"WRDS/{WRDS_DATABASE.upper()}: {', '.join(WRDS_COMPANIES)} ({WRDS_START_YEAR}-{WRDS_END_YEAR})"
    
    print("\n" + "-" * 60)
    print("✅ WRDS CONFIGURATION COMPLETE")
    print(f"   Database: {WRDS_DATABASE.upper()}")
    print(f"   Analysis: {analysis_type.upper()}")
    print(f"   Companies: {', '.join(WRDS_COMPANIES)}")
    print(f"   Years: {WRDS_START_YEAR} - {WRDS_END_YEAR}")
    print(f"   Title: {COMPANY_NAME}")
    print("-" * 60)

print("\n" + "=" * 80)
print("✅ Configuration complete!")
print(f"📊 Data source: {DATA_SOURCE}")
print(f"📈 Report title: {COMPANY_NAME}")
print("=" * 80)
print("\n👉 NEXT: Run Cell 3 to load your data")

## 4. Data Loading

- **Kaggle mode**: Reads local Excel or CSV file
- **WRDS mode**: Connects to WRDS, downloads Compustat or CRSP data

WRDS mode requires username and password (password input is hidden for security).

In [ ]:
# ============================================
# DATA LOADING
# ============================================

print("\n" + "=" * 80)
print("                    DATA LOADING")
print("=" * 80)

# ========== KAGGLE MODE ==========
if DATA_SOURCE == 'kaggle':
    print(f"\n📁 Loading file: {KAGGLE_FILE}")
    
    if KAGGLE_FILE.endswith('.xlsx'):
        df = pd.read_excel(KAGGLE_FILE)
    else:
        df = pd.read_csv(KAGGLE_FILE, encoding='latin1')
    
    print(f"✅ Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")

# ========== WRDS MODE ==========
elif DATA_SOURCE == 'wrds':
    print("\n🔐 WRDS LOGIN")
    print("-" * 40)
    
    # Get username
    wrds_username = input("👉 WRDS username: ")
    
    # Get password (hidden)
    wrds_password = getpass.getpass("👉 WRDS password: ")
    
    print("\n🔄 Connecting to WRDS...")
    
    # Connect to WRDS
    db = wrds.Connection(wrds_username=wrds_username, wrds_password=wrds_password)
    print("✅ Connected to WRDS")
    
    # Build company list string
    companies_str = "', '".join(WRDS_COMPANIES)
    
    print(f"\n📊 Downloading Compustat data...")
    print(f"   Companies: {', '.join(WRDS_COMPANIES)}")
    print(f"   Years: {WRDS_START_YEAR}-{WRDS_END_YEAR}")
    print("   ⏳ Please wait...")
    
    # Query Compustat data
    df = db.raw_sql(f"""
        SELECT 
            gvkey, datadate, fyear, tic as ticker, conm as company_name,
            sale as revenue, cogs as cost_of_sales, xsga as sg_and_a,
            oiadp as operating_income, ib as net_income,
            at as total_assets, act as current_assets,
            lct as current_liabilities, dltt as long_term_debt,
            seq as shareholders_equity, oancf as operating_cf, capx
        FROM comp.funda
        WHERE indfmt = 'INDL' AND datafmt = 'STD' 
            AND popsrc = 'D' AND consol = 'C'
            AND fyear BETWEEN {WRDS_START_YEAR} AND {WRDS_END_YEAR}
            AND tic IN ('{companies_str}')
        ORDER BY ticker, fyear
    """)
    
    # Close connection
    db.close()
    print("✅ WRDS connection closed")
    
    if len(df) == 0:
        print("\n❌ No data found! Check your tickers.")
        raise ValueError("No data returned from WRDS")
    
    print(f"✅ Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
    
    # Show companies found
    companies_found = df['ticker'].unique().tolist()
    print("\n📋 Companies with data:")
    for ticker in companies_found:
        rows = len(df[df['ticker'] == ticker])
        print(f"   {ticker}: {rows} records")

print("\n" + "=" * 80)
print("✅ Data loaded successfully!")
print(f"📈 Report title: {COMPANY_NAME}")
print("=" * 80)

print("\n📋 First 5 rows of data:")
print(df.head())

## 5. Column Name Standardization

**Purpose**: Unify column names from different data sources to standard names.

**Key features**:
- Convert all column names to lowercase (handles case sensitivity)
- Map common column names to standard names (e.g., Sales → sales, Revenue → sales)
- Unmapped columns prompt user to add to USER_*_NAMES lists

**Supported column types**:
- sales, profit, cost (financial)
- category, region, segment (retail)
- date, year, quarter, month (time)
- ticker, company_name (multi-company)
- ret, prc, vol (stock)
- roa, roe, debt_ratio, current_ratio (ratios)
- total_assets, total_liabilities, equity (balance sheet)
- operating_cf, investing_cf, financing_cf (cash flow)
- risk_class, credit_score (risk)

In [ ]:
# ============================================
# ENHANCED COLUMN NAME MAPPING
# ============================================

print("=" * 80)
print("                    ENHANCED COLUMN MAPPING")
print("=" * 80)

# =========================================================
# UNIFY ALL COLUMN NAMES TO LOWERCASE
# =========================================================
df.columns = df.columns.str.lower()
print("✅ All column names converted to lowercase")

# =========================================================
# COMPLETE COLUMN MAPPING
# =========================================================

COLUMN_MAPPING = {
    # === Sales/Revenue ===
    'sales': [
        'sales', 'sale', 'revenue', 'revenues', 'total_revenue', 'total_revenues',
        'net_revenue', 'gross_revenue', 'sales_revenue', 'income', 'total_income',
        'gross_income', 'turnover', 'total_sales', 'net_sales', 'gross_sales'
    ],
    
    # === Profit/Net Income ===
    'profit': [
        'profit', 'profits', 'net_income', 'netincome', 'net_profit', 'netprofit',
        'gross_profit', 'grossprofit', 'operating_profit', 'earnings', 'net_earnings',
        'profit_before_tax', 'pbt', 'profit_after_tax', 'pat', 'ni', 'ib'
    ],
    
    # === Cost ===
    'cost': [
        'cost', 'costs', 'total_cost', 'total_costs', 'cogs', 'cost_of_goods_sold',
        'cost_of_sales', 'operating_cost', 'operating_expense', 'opex', 'expenses',
        'total_expenses', 'sg&a', 'xsga'
    ],
    
    # === Discount ===
    'discount': [
        'discount', 'discounts', 'discount_rate', 'discount_pct',
        'discount_percent', 'discount%'
    ],
    
    # === Quantity ===
    'quantity': [
        'quantity', 'qty', 'order_quantity', 'units', 'volume'
    ],
    
    # === Product Category ===
    'category': [
        'category', 'categories', 'product_category', 'product_group', 'product_type',
        'item_category', 'class', 'product_class'
    ],
    
    # === Sub-Category ===
    'sub_category': [
        'sub_category', 'subcategory', 'sub-category', 'sub category',
        'product_subcategory', 'sub_cat', 'sub_product'
    ],
    
    # === Region ===
    'region': [
        'region', 'regions', 'sales_region', 'territory', 'area', 'zone', 'district'
    ],
    
    # === Customer Segment ===
    'segment': [
        'segment', 'segments', 'customer_segment', 'market_segment',
        'customer_type', 'client_type'
    ],
    
    # === Order/Customer IDs ===
    'order_id': [
        'order_id', 'orderid', 'order_number', 'transaction_id', 'invoice_id'
    ],
    'customer_id': [
        'customer_id', 'customerid', 'client_id', 'user_id'
    ],
    
    # === Date Fields ===
    'date': [
        'date', 'dates', 'transaction_date', 'trans_date', 'order_date',
        'invoice_date', 'report_date', 'fiscal_date', 'calendar_date', 'datadate'
    ],
    'ship_date': [
        'ship_date', 'shipdate', 'delivery_date', 'shipping_date'
    ],
    
    # === Time Features ===
    'year': [
        'year', 'years', 'fiscal_year', 'calendar_year', 'report_year', 'yr', 'fyear'
    ],
    'quarter': [
        'quarter', 'quarters', 'qtr', 'fiscal_quarter', 'calendar_quarter', 'qrt', 'q'
    ],
    'month': [
        'month', 'months', 'fiscal_month', 'calendar_month', 'period', 'mth'
    ],
    'week': [
        'week', 'weeks', 'week_number', 'week_num', 'fiscal_week'
    ],
    'day': [
        'day', 'days', 'day_of_month', 'day_of_week', 'dow', 'dom'
    ],
    
    # === Stock Ticker ===
    'ticker': [
        'ticker', 'tic', 'stock_code', 'symbol', 'stock_symbol', 'security_code',
        'company_id', 'companyid', 'firm_id', 'company_code', 'comp_id', 'entity_id'
    ],
    
    # === Company Name ===
    'company_name': [
        'company_name', 'companyname', 'conm', 'company', 'firm_name', 'corporation',
        'comp_name', 'entity_name'
    ],
    
    # === Stock Market Data ===
    'ret': [
        'ret', 'return', 'returns', 'daily_return', 'dailyret', 'stock_return', 'total_return'
    ],
    'prc': [
        'prc', 'price', 'close', 'close_price', 'adj_close', 'adjusted_close', 'stock_price'
    ],
    'vol': [
        'vol', 'volume', 'trading_volume', 'share_volume', 'daily_volume'
    ],
    
    # === Financial Ratios ===
    'roa': [
        'roa', 'return_on_assets', 'returnonassets', 'asset_return', 'asset_turnover'
    ],
    'roe': [
        'roe', 'return_on_equity', 'returnonequity', 'equity_return', 'shareholder_return'
    ],
    'debt_ratio': [
        'debt_ratio', 'debt_to_assets', 'debttoassets', 'leverage', 'debt_to_equity'
    ],
    'current_ratio': [
        'current_ratio', 'liquidity_ratio', 'quick_ratio', 'working_capital_ratio'
    ],
    
    # === Gross Margin ===
    'gross_margin': [
        'gross_margin', 'grossmargin', 'gross_profit_margin', 'gpm'
    ],
    
    # === Transaction Volume ===
    'transaction_volume': [
        'transaction_volume', 'transactionvolume', 'txn_volume', 'trade_volume'
    ],
    
    # === Market Volatility (正确的映射) ===
    'market_volatility': [
        'market_volatility_index', 'market_volatility', 'marketvolatility', 
        'volatility_index', 'vix', 'market_vol'
    ],
    
    # === Management Score (正确的映射) ===
    'management_score': [
        'management_activity_score', 'management_score', 'managementscore', 
        'mgmt_score', 'activity_score'
    ],
    
    # === Assets and Liabilities ===
    'total_assets': [
        'total_assets', 'totalassets', 'assets', 'asset', 'asset_total', 'at'
    ],
    'current_assets': [
        'current_assets', 'currentassets', 'cur_assets', 'ca', 'act'
    ],
    'total_liabilities': [
        'total_liabilities', 'totalliabilities', 'liabilities', 'liab', 'total_debt', 'lt'
    ],
    'current_liabilities': [
        'current_liabilities', 'currentliabilities', 'cur_liabilities', 'cl', 'lct'
    ],
    'long_term_debt': [
        'long_term_debt', 'longtermdebt', 'lt_debt', 'long_term_liabilities', 'dltt'
    ],
    'shareholders_equity': [
        'shareholders_equity', 'shareholderequity', 'stockholders_equity', 'total_equity',
        'equity', 'book_value', 'seq'
    ],
    
    # === Cash Flow ===
    'operating_cf': [
        'operating_cf', 'operating_cash_flow', 'operatingcashflow', 'ocf', 'cfo',
        'cash_flow_operating', 'oancf', 'cash_flow', 'cashflow'
    ],
    'investing_cf': [
        'investing_cf', 'investing_cash_flow', 'investingcashflow', 'icf', 'cfi', 'ivncf'
    ],
    'financing_cf': [
        'financing_cf', 'financing_cash_flow', 'financingcashflow', 'fcf', 'cff', 'fincf'
    ],
    'capital_expenditure': [
        'capital_expenditure', 'capex', 'capx', 'capital_spending'
    ],
    
    # === Risk and Credit (包含 risk_tag) ===
    'risk_class': [
        'risk_class', 'riskclass', 'risk_tag', 'risktag', 'risk_level', 'credit_rating'
    ],
    'credit_score': [
        'credit_score', 'creditscore', 'credit_rating'
    ]
}

# =========================================================
# USER CUSTOM MAPPINGS
# =========================================================
USER_SALES_NAMES = []
USER_PROFIT_NAMES = []
USER_COST_NAMES = []
USER_DISCOUNT_NAMES = []
USER_QUANTITY_NAMES = []
USER_CATEGORY_NAMES = []
USER_SUB_CATEGORY_NAMES = []
USER_REGION_NAMES = []
USER_SEGMENT_NAMES = []
USER_TICKER_NAMES = []
USER_DATE_NAMES = []
USER_YEAR_NAMES = []
USER_QUARTER_NAMES = []
USER_MONTH_NAMES = []
USER_RET_NAMES = []
USER_PRC_NAMES = []
USER_ROA_NAMES = []
USER_ROE_NAMES = []
USER_DEBT_NAMES = []
USER_ASSETS_NAMES = []
USER_LIABILITIES_NAMES = []
USER_EQUITY_NAMES = []
USER_GROSS_MARGIN_NAMES = []
USER_TRANSACTION_VOLUME_NAMES = []
USER_MARKET_VOLATILITY_NAMES = []
USER_MANAGEMENT_SCORE_NAMES = []
USER_RISK_CLASS_NAMES = []
USER_CASH_FLOW_NAMES = []

# Merge user custom names
if USER_SALES_NAMES:
    COLUMN_MAPPING['sales'].extend(USER_SALES_NAMES)
if USER_PROFIT_NAMES:
    COLUMN_MAPPING['profit'].extend(USER_PROFIT_NAMES)
if USER_COST_NAMES:
    COLUMN_MAPPING['cost'].extend(USER_COST_NAMES)
if USER_DISCOUNT_NAMES:
    COLUMN_MAPPING['discount'].extend(USER_DISCOUNT_NAMES)
if USER_QUANTITY_NAMES:
    COLUMN_MAPPING['quantity'].extend(USER_QUANTITY_NAMES)
if USER_CATEGORY_NAMES:
    COLUMN_MAPPING['category'].extend(USER_CATEGORY_NAMES)
if USER_SUB_CATEGORY_NAMES:
    COLUMN_MAPPING['sub_category'].extend(USER_SUB_CATEGORY_NAMES)
if USER_REGION_NAMES:
    COLUMN_MAPPING['region'].extend(USER_REGION_NAMES)
if USER_SEGMENT_NAMES:
    COLUMN_MAPPING['segment'].extend(USER_SEGMENT_NAMES)
if USER_TICKER_NAMES:
    COLUMN_MAPPING['ticker'].extend(USER_TICKER_NAMES)
if USER_DATE_NAMES:
    COLUMN_MAPPING['date'].extend(USER_DATE_NAMES)
if USER_YEAR_NAMES:
    COLUMN_MAPPING['year'].extend(USER_YEAR_NAMES)
if USER_QUARTER_NAMES:
    COLUMN_MAPPING['quarter'].extend(USER_QUARTER_NAMES)
if USER_MONTH_NAMES:
    COLUMN_MAPPING['month'].extend(USER_MONTH_NAMES)
if USER_RET_NAMES:
    COLUMN_MAPPING['ret'].extend(USER_RET_NAMES)
if USER_PRC_NAMES:
    COLUMN_MAPPING['prc'].extend(USER_PRC_NAMES)
if USER_ROA_NAMES:
    COLUMN_MAPPING['roa'].extend(USER_ROA_NAMES)
if USER_ROE_NAMES:
    COLUMN_MAPPING['roe'].extend(USER_ROE_NAMES)
if USER_DEBT_NAMES:
    COLUMN_MAPPING['debt_ratio'].extend(USER_DEBT_NAMES)
if USER_ASSETS_NAMES:
    COLUMN_MAPPING['total_assets'].extend(USER_ASSETS_NAMES)
if USER_LIABILITIES_NAMES:
    COLUMN_MAPPING['total_liabilities'].extend(USER_LIABILITIES_NAMES)
if USER_EQUITY_NAMES:
    COLUMN_MAPPING['shareholders_equity'].extend(USER_EQUITY_NAMES)
if USER_GROSS_MARGIN_NAMES:
    COLUMN_MAPPING['gross_margin'].extend(USER_GROSS_MARGIN_NAMES)
if USER_TRANSACTION_VOLUME_NAMES:
    COLUMN_MAPPING['transaction_volume'].extend(USER_TRANSACTION_VOLUME_NAMES)
if USER_MARKET_VOLATILITY_NAMES:
    COLUMN_MAPPING['market_volatility'].extend(USER_MARKET_VOLATILITY_NAMES)
if USER_MANAGEMENT_SCORE_NAMES:
    COLUMN_MAPPING['management_score'].extend(USER_MANAGEMENT_SCORE_NAMES)
if USER_RISK_CLASS_NAMES:
    COLUMN_MAPPING['risk_class'].extend(USER_RISK_CLASS_NAMES)
if USER_CASH_FLOW_NAMES:
    COLUMN_MAPPING['operating_cf'].extend(USER_CASH_FLOW_NAMES)

# =========================================================
# DETECT AND MAP COLUMNS
# =========================================================
available_cols = {}
for standard_name, possible_names in COLUMN_MAPPING.items():
    for col in possible_names:
        if col in df.columns:
            available_cols[standard_name] = col
            break

print(f"\n✅ Detected and mapped columns:")
for standard_name, original_col in available_cols.items():
    print(f"   {standard_name} <- '{original_col}'")

print(f"\n📊 Summary:")
print(f"   Total columns in data: {len(df.columns)}")
print(f"   Mapped columns: {len(available_cols)}")

# Show unmapped columns
unmapped = [col for col in df.columns if col not in available_cols.values()]
if unmapped:
    print(f"\n⚠️ Unmapped columns (not used in analysis):")
    for col in unmapped[:15]:
        print(f"   - {col}")
    if len(unmapped) > 15:
        print(f"   ... and {len(unmapped) - 15} more")
    print("\n💡 TIP: To include these columns, add them to the USER_*_NAMES lists above")

# Rename columns
rename_dict = {}
for standard_name, original_col in available_cols.items():
    if standard_name != original_col:
        rename_dict[original_col] = standard_name

if rename_dict:
    df.rename(columns=rename_dict, inplace=True)
    print(f"\n✅ Renamed {len(rename_dict)} columns: {list(rename_dict.values())}")
else:
    print("\n✅ No renaming needed")

print("\n✅ Column mapping complete!")

## 6. Data Cleaning

**Cleaning steps**:

1. Data type conversion (numeric, datetime)
2. Missing value analysis and filling (median for numeric, mode for categorical)
3. Duplicate row removal
4. Outlier detection (IQR method, 3x IQR threshold)
5. Logical consistency checks (profit ≤ sales, discount ≥ 0)
6. Data quality report

In [ ]:
# ============================================
# COMPREHENSIVE DATA CLEANING (UNIVERSAL FIX)
# Solves: datetime, NameError, division by zero, missing columns
# ============================================

print("=" * 80)
print("                    COMPREHENSIVE DATA CLEANING")
print("=" * 80)

# =========================================================
# DEFINE ALL VARIABLES (FIXES NameError)
# =========================================================
is_multi_company = 'ticker' in df.columns and len(df['ticker'].unique()) > 1
is_stock_data = 'ret' in df.columns or 'prc' in df.columns
is_financial_data = 'profit' in df.columns or 'sales' in df.columns
is_retail_data = 'category' in df.columns or 'region' in df.columns or 'segment' in df.columns

original_count = len(df)
original_cols = len(df.columns)

print(f"\n📊 Data Type Detection:")
print(f"   Multi-company: {'Yes' if is_multi_company else 'No'}")
print(f"   Stock data: {'Yes' if is_stock_data else 'No'}")
print(f"   Financial data: {'Yes' if is_financial_data else 'No'}")
print(f"   Retail data: {'Yes' if is_retail_data else 'No'}")

# =========================================================
# 1. COLUMN NAME STANDARDIZATION
# =========================================================
print("\n" + "-" * 60)
print("1. Column Name Standardization")
print("-" * 60)

print("\nFinal column names:")
for col in df.columns:
    print(f"   {col}")

# =========================================================
# 2. DATE COLUMN CONVERSION (FIXES .dt accessor error)
# =========================================================
print("\n" + "-" * 60)
print("2. Date Column Conversion")
print("-" * 60)

# List of possible date column names
date_column_candidates = ['datadate', 'date', 'order_date', 'Order Date', 'Date', 'fiscal_date']

date_converted = False
for date_col in date_column_candidates:
    if date_col in df.columns:
        original_dtype = df[date_col].dtype
        try:
            # Try different conversion methods
            if df[date_col].dtype == 'int64' or df[date_col].dtype == 'float64':
                # Convert integer dates like 20201231 to datetime
                df[date_col] = pd.to_datetime(df[date_col].astype(str), format='%Y%m%d', errors='coerce')
            else:
                df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
            
            # Check if conversion worked
            if df[date_col].isnull().sum() < len(df):
                # Rename to standard 'date' column
                df.rename(columns={date_col: 'date'}, inplace=True)
                print(f"   ✅ Converted '{date_col}' from {original_dtype} to datetime")
                date_converted = True
                break
        except:
            print(f"   ⚠️ Could not convert '{date_col}' to datetime")
            continue

if not date_converted:
    print("   ℹ️ No date column found or conversion failed. Time features will be skipped.")

# =========================================================
# 3. NUMERIC COLUMN CONVERSION
# =========================================================
print("\n" + "-" * 60)
print("3. Numeric Column Conversion")
print("-" * 60)

numeric_candidates = ['sales', 'profit', 'cost_of_sales', 'sg_and_a', 'operating_income',
                      'total_assets', 'long_term_debt', 'debt_ratio', 'roa', 'roe',
                      'revenue', 'net_income', 'discount', 'quantity', 'ret', 'prc', 'vol']

for col in numeric_candidates:
    if col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], errors='coerce')
            print(f"   ✅ {col} converted to numeric")

# =========================================================
# 4. MISSING VALUE ANALYSIS
# =========================================================
print("\n" + "-" * 60)
print("4. Missing Value Analysis")
print("-" * 60)

missing_count = df.isnull().sum()
missing_percent = (missing_count / len(df)) * 100
missing_df = pd.DataFrame({'Missing': missing_count, 'Percent': missing_percent})
missing_df = missing_df[missing_df['Missing'] > 0].sort_values('Percent', ascending=False)

if len(missing_df) > 0:
    print("\nMissing values:")
    print(missing_df)
    
    # Drop columns with >50% missing
    high_missing = missing_df[missing_df['Percent'] > 50].index.tolist()
    if high_missing:
        print(f"\n⚠️ Dropping columns with >50% missing: {high_missing}")
        df = df.drop(columns=high_missing)
    
    # Fill remaining numeric missing values with median
    print("\nFilling missing values:")
    for col in df.columns:
        if df[col].dtype in ['float64', 'int64'] and df[col].isnull().sum() > 0:
            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)
            print(f"   {col}: filled with median {median_val:.4f}")
        elif df[col].dtype == 'object' and df[col].isnull().sum() > 0:
            mode_val = df[col].mode()[0] if len(df[col].mode()) > 0 else 'Unknown'
            df[col].fillna(mode_val, inplace=True)
            print(f"   {col}: filled with mode '{mode_val}'")
else:
    print("   ✅ No missing values")

# =========================================================
# 5. DUPLICATE REMOVAL
# =========================================================
print("\n" + "-" * 60)
print("5. Duplicate Removal")
print("-" * 60)

duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")
if duplicates > 0:
    df = df.drop_duplicates()
    print(f"   ✅ Removed {duplicates} duplicates")

# =========================================================
# 6. OUTLIER DETECTION (with safe defaults)
# =========================================================
print("\n" + "-" * 60)
print("6. Outlier Detection")
print("-" * 60)

outlier_cols = [c for c in ['sales', 'profit', 'debt_ratio', 'roa', 'roe', 'ret'] if c in df.columns]

for col in outlier_cols:
    if col in df.columns and len(df) > 5:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 3 * IQR
        upper = Q3 + 3 * IQR
        
        outliers = df[(df[col] < lower) | (df[col] > upper)]
        if len(outliers) > 0 and len(outliers) < len(df) * 0.1:  # Only fix if <10% outliers
            p1 = df[col].quantile(0.01)
            p99 = df[col].quantile(0.99)
            df.loc[df[col] < lower, col] = p1
            df.loc[df[col] > upper, col] = p99
            print(f"   ✅ {col}: fixed {len(outliers)} outliers")

# =========================================================
# 7. LOGICAL CONSISTENCY (with division by zero protection)
# =========================================================
print("\n" + "-" * 60)
print("7. Logical Consistency")
print("-" * 60)

# Safe profit margin calculation
if 'profit' in df.columns and 'sales' in df.columns:
    # Avoid division by zero
    df['profit_margin'] = np.where(
        df['sales'] != 0,
        (df['profit'] / df['sales']) * 100,
        0
    )
    df['profit_margin'] = df['profit_margin'].fillna(0)
    print("   ✅ Calculated profit_margin with division by zero protection")

# Fix profit > sales
if 'profit' in df.columns and 'sales' in df.columns:
    bad = df[abs(df['profit']) > df['sales']]
    if len(bad) > 0:
        df.loc[abs(df['profit']) > df['sales'], 'profit'] = df['sales'] * 0.8
        print(f"   ✅ Fixed {len(bad)} records where profit > sales")

# Create loss flag
if 'profit' in df.columns:
    df['is_loss'] = df['profit'] < 0
    print("   ✅ Created is_loss flag")

# =========================================================
# 8. DATA QUALITY REPORT
# =========================================================
print("\n" + "-" * 60)
print("8. Data Quality Report")
print("-" * 60)

final_rows = len(df)
final_cols = len(df.columns)
rows_removed = original_count - final_rows
pct = (rows_removed / original_count) * 100 if original_count > 0 else 0

print(f"\n   Rows: {original_count} → {final_rows} (removed {rows_removed}, {pct:.1f}%)")
print(f"   Columns: {original_cols} → {final_cols}")
print(f"   Missing values remaining: {df.isnull().sum().sum()}")

print("\n" + "=" * 80)
print("✅ DATA CLEANING COMPLETE!")
print("=" * 80)

## 7. Feature Engineering

**Created features**:

- `profit_margin`: Profit/Sales × 100%
- `profit_level`: A-E classification based on margin
- `is_loss`: Loss flag (profit < 0)
- `loss_severity`: Mild / Moderate / Severe loss classification
- `roa`, `roe`: Return on Assets / Equity
- `debt_ratio`: Debt to Assets ratio
- `current_ratio`: Current Assets / Current Liabilities
- `discount_level`: Discount classification (retail data)
- `order_size`: Order size classification (retail data)
- `return_level`, `volatility`, `cumulative_ret` (stock data)
- `category_region`, `segment_region`: Cross features
- `financial_health_score`: Composite score (0-100)



In [ ]:
# ============================================
# COMPREHENSIVE FEATURE ENGINEERING
# ============================================

print("=" * 80)
print("                    COMPREHENSIVE FEATURE ENGINEERING")
print("=" * 80)

features_created = []
original_feature_count = len(df.columns)

# Detect data type
is_multi_company = 'ticker' in df.columns and len(df['ticker'].unique()) > 1
is_stock_data = 'ret' in df.columns or 'prc' in df.columns
is_financial_data = 'profit' in df.columns or 'sales' in df.columns

print(f"\n📊 Data Type Detection:")
print(f"   Multi-company: {'Yes' if is_multi_company else 'No'}")
print(f"   Stock data: {'Yes' if is_stock_data else 'No'}")
print(f"   Financial data: {'Yes' if is_financial_data else 'No'}")

# =========================================================
# 1. Profitability Features
# =========================================================
if is_financial_data:
    print("\n" + "-" * 60)
    print("1. Profitability Features")
    print("-" * 60)
    
    # Gross Profit Margin
    if 'profit' in df.columns and 'sales' in df.columns:
        df['gross_margin'] = (df['profit'] / df['sales']) * 100
        df['gross_margin'] = df['gross_margin'].replace([np.inf, -np.inf], 0).fillna(0)
        features_created.append('gross_margin')
        print(f"   ✅ gross_margin: {df['gross_margin'].mean():.2f}% avg")
    
    # Operating Margin (if operating income available)
    if 'operating_income' in df.columns and 'sales' in df.columns:
        df['operating_margin'] = (df['operating_income'] / df['sales']) * 100
        df['operating_margin'] = df['operating_margin'].replace([np.inf, -np.inf], 0).fillna(0)
        features_created.append('operating_margin')
        print(f"   ✅ operating_margin: {df['operating_margin'].mean():.2f}% avg")
    
    # Net Profit Margin
    if 'net_income' in df.columns and 'sales' in df.columns:
        df['net_margin'] = (df['net_income'] / df['sales']) * 100
        df['net_margin'] = df['net_margin'].replace([np.inf, -np.inf], 0).fillna(0)
        features_created.append('net_margin')
        print(f"   ✅ net_margin: {df['net_margin'].mean():.2f}% avg")
    
    # EBIT Margin (if available)
    if 'operating_income' in df.columns and 'sales' in df.columns:
        df['ebit_margin'] = (df['operating_income'] / df['sales']) * 100
        df['ebit_margin'] = df['ebit_margin'].replace([np.inf, -np.inf], 0).fillna(0)
        features_created.append('ebit_margin')
        print(f"   ✅ ebit_margin: {df['ebit_margin'].mean():.2f}% avg")

# =========================================================
# 2. Efficiency Ratios
# =========================================================
if is_financial_data:
    print("\n" + "-" * 60)
    print("2. Efficiency Ratios")
    print("-" * 60)
    
    # Asset Turnover
    if 'sales' in df.columns and 'total_assets' in df.columns:
        df['asset_turnover'] = df['sales'] / df['total_assets']
        df['asset_turnover'] = df['asset_turnover'].replace([np.inf, -np.inf], 0).fillna(0)
        features_created.append('asset_turnover')
        print(f"   ✅ asset_turnover: {df['asset_turnover'].mean():.2f} avg")
    
    # ROA (Return on Assets)
    if 'profit' in df.columns and 'total_assets' in df.columns:
        df['roa'] = (df['profit'] / df['total_assets']) * 100
        df['roa'] = df['roa'].replace([np.inf, -np.inf], 0).fillna(0)
        if 'roa' not in features_created:
            features_created.append('roa')
        print(f"   ✅ roa: {df['roa'].mean():.2f}% avg return on assets")
    
    # ROE (Return on Equity)
    if 'profit' in df.columns and 'shareholders_equity' in df.columns:
        df['roe'] = (df['profit'] / df['shareholders_equity']) * 100
        df['roe'] = df['roe'].replace([np.inf, -np.inf], 0).fillna(0)
        if 'roe' not in features_created:
            features_created.append('roe')
        print(f"   ✅ roe: {df['roe'].mean():.2f}% avg return on equity")
    
    # ROCE (Return on Capital Employed)
    if 'operating_income' in df.columns and 'total_assets' in df.columns and 'current_liabilities' in df.columns:
        capital_employed = df['total_assets'] - df['current_liabilities']
        df['roce'] = (df['operating_income'] / capital_employed) * 100
        df['roce'] = df['roce'].replace([np.inf, -np.inf], 0).fillna(0)
        features_created.append('roce')
        print(f"   ✅ roce: {df['roce'].mean():.2f}% avg return on capital employed")

# =========================================================
# 3. Leverage and Solvency Ratios
# =========================================================
if is_financial_data:
    print("\n" + "-" * 60)
    print("3. Leverage and Solvency Ratios")
    print("-" * 60)
    
    # Debt to Equity
    if 'long_term_debt' in df.columns and 'shareholders_equity' in df.columns:
        df['debt_to_equity'] = (df['long_term_debt'] / df['shareholders_equity']) * 100
        df['debt_to_equity'] = df['debt_to_equity'].replace([np.inf, -np.inf], 0).fillna(0)
        features_created.append('debt_to_equity')
        print(f"   ✅ debt_to_equity: {df['debt_to_equity'].mean():.2f}% avg")
    
    # Debt to Assets
    if 'long_term_debt' in df.columns and 'total_assets' in df.columns:
        df['debt_to_assets'] = (df['long_term_debt'] / df['total_assets']) * 100
        df['debt_to_assets'] = df['debt_to_assets'].replace([np.inf, -np.inf], 0).fillna(0)
        if 'debt_ratio' not in features_created:
            features_created.append('debt_to_assets')
        print(f"   ✅ debt_to_assets: {df['debt_to_assets'].mean():.2f}% avg")
    
    # Interest Coverage Ratio (if available)
    if 'operating_income' in df.columns and 'interest_expense' in df.columns:
        df['interest_coverage'] = df['operating_income'] / df['interest_expense'].replace(0, np.nan)
        df['interest_coverage'] = df['interest_coverage'].replace([np.inf, -np.inf], 0).fillna(0)
        features_created.append('interest_coverage')
        print(f"   ✅ interest_coverage: {df['interest_coverage'].mean():.2f} avg")

# =========================================================
# 4. Liquidity Ratios
# =========================================================
if is_financial_data:
    print("\n" + "-" * 60)
    print("4. Liquidity Ratios")
    print("-" * 60)
    
    # Current Ratio
    if 'current_assets' in df.columns and 'current_liabilities' in df.columns:
        df['current_ratio'] = df['current_assets'] / df['current_liabilities']
        df['current_ratio'] = df['current_ratio'].replace([np.inf, -np.inf], 0).fillna(0)
        features_created.append('current_ratio')
        print(f"   ✅ current_ratio: {df['current_ratio'].mean():.2f} avg")
        
        # Quick Ratio (excluding inventory)
        if 'inventory' in df.columns:
            df['quick_ratio'] = (df['current_assets'] - df['inventory']) / df['current_liabilities']
            df['quick_ratio'] = df['quick_ratio'].replace([np.inf, -np.inf], 0).fillna(0)
            features_created.append('quick_ratio')
            print(f"   ✅ quick_ratio: {df['quick_ratio'].mean():.2f} avg")
    
    # Cash Ratio
    if 'cash' in df.columns and 'current_liabilities' in df.columns:
        df['cash_ratio'] = df['cash'] / df['current_liabilities']
        df['cash_ratio'] = df['cash_ratio'].replace([np.inf, -np.inf], 0).fillna(0)
        features_created.append('cash_ratio')
        print(f"   ✅ cash_ratio: {df['cash_ratio'].mean():.2f} avg")

# =========================================================
# 5. Cash Flow Ratios
# =========================================================
if 'operating_cf' in df.columns:
    print("\n" + "-" * 60)
    print("5. Cash Flow Ratios")
    print("-" * 60)
    
    # Operating Cash Flow to Sales
    if 'sales' in df.columns:
        df['cf_to_sales'] = (df['operating_cf'] / df['sales']) * 100
        df['cf_to_sales'] = df['cf_to_sales'].replace([np.inf, -np.inf], 0).fillna(0)
        features_created.append('cf_to_sales')
        print(f"   ✅ cf_to_sales: {df['cf_to_sales'].mean():.2f}% avg")
    
    # Operating Cash Flow to Net Income
    if 'net_income' in df.columns:
        df['cf_to_ni'] = df['operating_cf'] / df['net_income'].replace(0, np.nan)
        df['cf_to_ni'] = df['cf_to_ni'].replace([np.inf, -np.inf], 0).fillna(0)
        features_created.append('cf_to_ni')
        print(f"   ✅ cf_to_ni: {df['cf_to_ni'].mean():.2f} avg")

# =========================================================
# 6. Classification Features
# =========================================================
print("\n" + "-" * 60)
print("6. Classification Features")
print("-" * 60)

# Profit Level (A-E scale)
if 'gross_margin' in df.columns:
    def profit_level(margin):
        if pd.isna(margin): return 'Unknown'
        elif margin < 0: return 'E (Loss)'
        elif margin < 5: return 'D (Poor)'
        elif margin < 10: return 'C (Average)'
        elif margin < 20: return 'B (Good)'
        else: return 'A (Excellent)'
    df['profit_level'] = df['gross_margin'].apply(profit_level)
    features_created.append('profit_level')
    print("   ✅ profit_level: A-E classification created")

# Loss Flag
if 'profit' in df.columns:
    df['is_loss'] = df['profit'] < 0
    features_created.append('is_loss')
    print(f"   ✅ is_loss: {df['is_loss'].sum()} loss-making records")

# Loss Severity
if 'profit' in df.columns:
    def loss_severity(profit):
        if profit >= 0: return 'Profitable'
        elif profit >= -50: return 'Mild Loss'
        elif profit >= -200: return 'Moderate Loss'
        else: return 'Severe Loss'
    df['loss_severity'] = df['profit'].apply(loss_severity)
    features_created.append('loss_severity')
    print("   ✅ loss_severity: loss severity classification")

# Discount Level
if 'discount' in df.columns:
    def discount_level(d):
        if pd.isna(d) or d == 0: return 'No Discount'
        elif d < 0.05: return 'Very Low (0-5%)'
        elif d < 0.1: return 'Low (5-10%)'
        elif d < 0.15: return 'Medium Low (10-15%)'
        elif d < 0.2: return 'Medium (15-20%)'
        elif d < 0.3: return 'High (20-30%)'
        else: return 'Very High (>30%)'
    df['discount_level'] = df['discount'].apply(discount_level)
    features_created.append('discount_level')
    print("   ✅ discount_level: 6-level discount classification")

# Order Size
if 'sales' in df.columns and not is_stock_data:
    def order_size(s):
        if pd.isna(s): return 'Unknown'
        elif s < 50: return 'Tiny (<$50)'
        elif s < 100: return 'Small ($50-100)'
        elif s < 250: return 'Medium ($100-250)'
        elif s < 500: return 'Large ($250-500)'
        else: return 'Very Large (>$500)'
    df['order_size'] = df['sales'].apply(order_size)
    features_created.append('order_size')
    print("   ✅ order_size: 5-level order size classification")

# =========================================================
# 7. Stock Market Features
# =========================================================
if is_stock_data:
    print("\n" + "-" * 60)
    print("7. Stock Market Features")
    print("-" * 60)
    
    # Return classification
    if 'ret' in df.columns:
        def return_level(r):
            if pd.isna(r): return 'Unknown'
            elif r > 0.03: return 'Strong Up (>3%)'
            elif r > 0: return 'Slight Up (0-3%)'
            elif r > -0.03: return 'Slight Down (-3%-0)'
            else: return 'Strong Down (<-3%)'
        df['return_level'] = df['ret'].apply(return_level)
        features_created.append('return_level')
        print("   ✅ return_level: daily return classification")
    
    # Cumulative return (per company)
    if 'ret' in df.columns and 'ticker' in df.columns:
        df['cumulative_ret'] = df.groupby('ticker')['ret'].transform(lambda x: (1 + x.fillna(0)).cumprod() - 1)
        features_created.append('cumulative_ret')
        print("   ✅ cumulative_ret: cumulative return by company")
    
    # Rolling volatility (20-day)
    if 'ret' in df.columns and 'ticker' in df.columns:
        df['volatility_20d'] = df.groupby('ticker')['ret'].transform(
            lambda x: x.rolling(20, min_periods=5).std()
        )
        features_created.append('volatility_20d')
        print("   ✅ volatility_20d: 20-day rolling volatility")
    
    # Price level classification
    if 'prc' in df.columns:
        def price_level(p):
            if pd.isna(p): return 'Unknown'
            elif p > 500: return 'Ultra High (>$500)'
            elif p > 200: return 'Very High ($200-500)'
            elif p > 100: return 'High ($100-200)'
            elif p > 50: return 'Medium ($50-100)'
            else: return 'Low (<$50)'
        df['price_level'] = df['prc'].apply(price_level)
        features_created.append('price_level')
        print("   ✅ price_level: 6-level price classification")

# =========================================================
# 8. Time-Based Features
# =========================================================
date_col = None
for col in ['date', 'order_date', 'datadate']:
    if col in df.columns:
        date_col = col
        break

if date_col:
    print("\n" + "-" * 60)
    print("8. Time-Based Features")
    print("-" * 60)
    
    df['year'] = df[date_col].dt.year
    df['month'] = df[date_col].dt.month
    df['quarter'] = df[date_col].dt.quarter
    df['week'] = df[date_col].dt.isocalendar().week
    df['weekday'] = df[date_col].dt.dayofweek
    df['day_of_month'] = df[date_col].dt.day
    df['is_weekend'] = df['weekday'].isin([5, 6]).astype(int)
    
    features_created.extend(['year', 'month', 'quarter', 'week', 'weekday', 'day_of_month', 'is_weekend'])
    print("   ✅ Time features: year, month, quarter, week, weekday, day_of_month, is_weekend")

# =========================================================
# 9. Cross Features (Interaction Terms)
# =========================================================
print("\n" + "-" * 60)
print("9. Cross Features (Interaction Terms)")
print("-" * 60)

# Category-Region
if 'category' in df.columns and 'region' in df.columns:
    df['category_region'] = df['category'].astype(str) + '_' + df['region'].astype(str)
    features_created.append('category_region')
    print("   ✅ category_region: product-region interaction")

# Segment-Region
if 'segment' in df.columns and 'region' in df.columns:
    df['segment_region'] = df['segment'].astype(str) + '_' + df['region'].astype(str)
    features_created.append('segment_region')
    print("   ✅ segment_region: customer-region interaction")

# Category-Segment
if 'category' in df.columns and 'segment' in df.columns:
    df['category_segment'] = df['category'].astype(str) + '_' + df['segment'].astype(str)
    features_created.append('category_segment')
    print("   ✅ category_segment: product-customer interaction")

# Year-Quarter composite (for time series analysis)
if 'year' in df.columns and 'quarter' in df.columns:
    df['year_quarter'] = df['year'].astype(str) + '-Q' + df['quarter'].astype(str)
    features_created.append('year_quarter')
    print("   ✅ year_quarter: year-quarter composite")

# =========================================================
# 10. Composite Scores
# =========================================================
print("\n" + "-" * 60)
print("10. Composite Scores")
print("-" * 60)

# Financial Health Score (0-100)
if 'gross_margin' in df.columns and 'is_loss' in df.columns:
    def financial_health_score(row):
        score = 50
        
        margin = row.get('gross_margin', 0)
        if margin >= 25: score += 30
        elif margin >= 15: score += 20
        elif margin >= 5: score += 10
        elif margin >= 0: score += 5
        else: score -= 15
        
        if row.get('is_loss', False): score -= 25
        
        debt = row.get('debt_to_assets', 50)
        if debt < 30: score += 15
        elif debt < 50: score += 5
        elif debt > 70: score -= 10
        
        discount = row.get('discount', 0)
        if discount <= 0.05: score += 10
        elif discount <= 0.1: score += 5
        elif discount > 0.2: score -= 10
        
        return max(0, min(100, score))
    
    df['financial_health_score'] = df.apply(financial_health_score, axis=1)
    features_created.append('financial_health_score')
    print(f"   ✅ financial_health_score: {df['financial_health_score'].mean():.1f} avg (0-100)")

# Profitability Score (0-100)
if 'gross_margin' in df.columns:
    df['profitability_score'] = df['gross_margin'].clip(0, 50) * 2
    features_created.append('profitability_score')
    print(f"   ✅ profitability_score: {df['profitability_score'].mean():.1f} avg")

# Leverage Score (0-100, lower is better)
if 'debt_to_assets' in df.columns:
    df['leverage_score'] = 100 - df['debt_to_assets'].clip(0, 100)
    features_created.append('leverage_score')
    print(f"   ✅ leverage_score: {df['leverage_score'].mean():.1f} avg")

# =========================================================
# 11. Ranking Features (for multi-company comparison)
# =========================================================
if is_multi_company and 'profit_margin' in df.columns:
    print("\n" + "-" * 60)
    print("11. Ranking Features")
    print("-" * 60)
    
    # Profit margin rank per year
    if 'year' in df.columns:
        df['profit_margin_rank'] = df.groupby('year')['profit_margin'].rank(ascending=False)
        features_created.append('profit_margin_rank')
        print("   ✅ profit_margin_rank: yearly ranking")
    
    # ROA rank
    if 'roa' in df.columns and 'year' in df.columns:
        df['roa_rank'] = df.groupby('year')['roa'].rank(ascending=False)
        features_created.append('roa_rank')
        print("   ✅ roa_rank: yearly ROA ranking")

# =========================================================
# 12. Summary
# =========================================================
print("\n" + "-" * 60)
print("12. Feature Engineering Summary")
print("-" * 60)

total_features_created = len(features_created)
total_cols = len(df.columns)

print(f"\n   Original features: {original_feature_count}")
print(f"   New features created: {total_features_created}")
print(f"   Total features now: {total_cols}")

print(f"\n   New features list:")
for i, feat in enumerate(features_created):
    print(f"      {i+1}. {feat}")

print("\n" + "=" * 80)
print("✅ COMPREHENSIVE FEATURE ENGINEERING COMPLETE!")
print("=" * 80)

## 8. Overall Financial Analysis

**Calculates**:

- Total Sales, Total Profit, Overall Profit Margin
- Multi-company comparison table by company (profit margin, ROA, debt ratio)
- Stock data: Average return, volatility, cumulative return

In [ ]:
# ============================================
# OVERALL FINANCIAL ANALYSIS
# ============================================

print("\n" + "=" * 80)
print(f"                    {COMPANY_NAME}")
print("                    OVERALL FINANCIAL ANALYSIS")
print("=" * 80)

# Overall metrics
if 'sales' in df.columns and 'profit' in df.columns:
    total_sales = df['sales'].sum()
    total_profit = df['profit'].sum()
    overall_margin = (total_profit / total_sales * 100) if total_sales > 0 else 0
    
    print(f"\n📊 Total Sales: ${total_sales:,.2f}")
    print(f"📊 Total Profit: ${total_profit:,.2f}")
    print(f"📊 Overall Profit Margin: {overall_margin:.2f}%")
    
    if total_profit > 0:
        print(f"\n✅ {COMPANY_NAME} is overall profitable")
    else:
        print(f"\n⚠️ {COMPANY_NAME} is overall unprofitable")

# Multi-company comparison (if multiple companies)
if 'ticker' in df.columns and len(df['ticker'].unique()) > 1:
    print("\n" + "=" * 80)
    print("                    MULTI-COMPANY COMPARISON")
    print("=" * 80)
    
    if 'profit_margin' in df.columns:
        company_margins = df.groupby('ticker')['profit_margin'].mean().sort_values(ascending=False)
        print("\n📊 Profit Margin by Company:")
        for ticker, margin in company_margins.items():
            print(f"   {ticker}: {margin:.2f}%")
    
    if 'roa' in df.columns:
        company_roa = df.groupby('ticker')['roa'].mean().sort_values(ascending=False)
        print("\n📊 Return on Assets (ROA) by Company:")
        for ticker, roa in company_roa.items():
            print(f"   {ticker}: {roa:.2f}%")
    
    if 'debt_ratio' in df.columns:
        company_debt = df.groupby('ticker')['debt_ratio'].mean().sort_values(ascending=False)
        print("\n📊 Debt Ratio by Company:")
        for ticker, debt in company_debt.items():
            print(f"   {ticker}: {debt:.2f}%")

# Financial health indicators
if 'roa' in df.columns:
    print(f"\n📊 Average Return on Assets (ROA): {df['roa'].mean():.2f}%")

if 'roe' in df.columns:
    print(f"📊 Average Return on Equity (ROE): {df['roe'].mean():.2f}%")

if 'debt_ratio' in df.columns:
    avg_debt = df['debt_ratio'].mean()
    print(f"📊 Average Debt to Assets Ratio: {avg_debt:.2f}%")
    if avg_debt > 50:
        print(f"   ⚠️ High leverage risk")
    elif avg_debt > 30:
        print(f"   📊 Moderate leverage")
    else:
        print(f"   ✅ Low leverage - conservative financing")

## 9. Dimensional Analysis

**Auto-detected analysis based on available columns**:

| Dimension | Analysis Content |
|-----------|------------------|
| Product Category | Profit by category |
| Region | Profit by region |
| Customer Segment | Profit by segment |
| Discount Level | Average profit by discount level |
| Year | Profit trend over years |
| Company (multi) | Profit margin, ROA, debt ratio by company |

In [ ]:
# ============================================
# DIMENSIONAL ANALYSIS
# ============================================

print("\n" + "=" * 80)
print("                    DIMENSIONAL ANALYSIS")
print("=" * 80)

# Profit level distribution
if 'profit_level' in df.columns:
    print("\n📊 Profit Level Distribution:")
    level_counts = df['profit_level'].value_counts()
    for level, count in level_counts.items():
        print(f"   {level}: {count} records ({count/len(df)*100:.1f}%)")

# Loss rate
if 'is_loss' in df.columns:
    loss_rate = (df['is_loss'].sum() / len(df)) * 100
    print(f"\n📊 Loss Rate: {loss_rate:.2f}%")

# Profit margin statistics
if 'profit_margin' in df.columns:
    print(f"\n📊 Profit Margin Statistics:")
    print(f"   Mean: {df['profit_margin'].mean():.2f}%")
    print(f"   Median: {df['profit_margin'].median():.2f}%")
    print(f"   Std Dev: {df['profit_margin'].std():.2f}%")
    print(f"   Min: {df['profit_margin'].min():.2f}%")
    print(f"   Max: {df['profit_margin'].max():.2f}%")

# Risk class distribution (if exists)
if 'risk_class' in df.columns:
    print("\n📊 Risk Class Distribution:")
    risk_counts = df['risk_class'].value_counts().sort_index()
    for risk, count in risk_counts.items():
        print(f"   Class {risk}: {count} records ({count/len(df)*100:.1f}%)")

# Category analysis (if exists)
if 'category' in df.columns and 'profit' in df.columns:
    print("\n📊 Profit by Category:")
    category_profit = df.groupby('category')['profit'].sum().sort_values(ascending=False)
    for cat, profit in category_profit.items():
        status = "✅" if profit > 0 else "⚠️"
        print(f"   {status} {cat}: ${profit:,.2f}")

# Region analysis (if exists)
if 'region' in df.columns and 'profit' in df.columns:
    print("\n📊 Profit by Region:")
    region_profit = df.groupby('region')['profit'].sum().sort_values(ascending=False)
    for reg, profit in region_profit.items():
        status = "✅" if profit > 0 else "⚠️"
        print(f"   {status} {reg}: ${profit:,.2f}")

## 10. Visualization

**Automatically generates 2-6 charts based on data type**:

| Data Type | Charts Generated |
|-----------|------------------|
| Multi-company financial | Profit margin bar chart, ROA bar chart, debt ratio bar chart, trend line |
| Single-company financial | Profit margin histogram, ROA histogram, loss pie chart, health score histogram |
| Stock data | Return distribution histogram, cumulative return line, volatility line |
| Retail data | Category profit bar chart, region profit bar chart, discount scatter plot |

**Output file**: `{company_name}_charts.png`

In [ ]:
# ============================================
# UNIVERSAL VISUALIZATION
# Auto-detects data and generates 2-6 charts
# ============================================

print("\n" + "=" * 80)
print("                    UNIVERSAL VISUALIZATION")
print("=" * 80)

# Clean filename for saving
clean_name = COMPANY_NAME.replace(":", "").replace("/", "_").replace(" ", "_").replace(",", "")
clean_name = clean_name.replace("(", "").replace(")", "").replace("&", "and")

# =========================================================
# DETECT AVAILABLE DATA
# =========================================================
has_multi = 'ticker' in df.columns and len(df['ticker'].unique()) > 1
has_stock = 'ret' in df.columns
has_profit_margin = 'profit_margin' in df.columns
has_roa = 'roa' in df.columns
has_debt = 'debt_ratio' in df.columns
has_loss = 'is_loss' in df.columns
has_health = 'financial_health_score' in df.columns or 'health_score' in df.columns
has_category = 'category' in df.columns and 'profit' in df.columns
has_region = 'region' in df.columns and 'profit' in df.columns
has_discount = 'discount' in df.columns and 'profit' in df.columns
has_year = 'year' in df.columns

print(f"\n📊 Available Data Detected:")
print(f"   Multi-company: {'Yes' if has_multi else 'No'}")
print(f"   Stock data: {'Yes' if has_stock else 'No'}")
print(f"   Profit margin: {'Yes' if has_profit_margin else 'No'}")
print(f"   ROA: {'Yes' if has_roa else 'No'}")
print(f"   Debt ratio: {'Yes' if has_debt else 'No'}")
print(f"   Loss flag: {'Yes' if has_loss else 'No'}")
print(f"   Health score: {'Yes' if has_health else 'No'}")
print(f"   Category: {'Yes' if has_category else 'No'}")
print(f"   Region: {'Yes' if has_region else 'No'}")
print(f"   Discount: {'Yes' if has_discount else 'No'}")

# =========================================================
# COLLECT AVAILABLE CHARTS
# =========================================================
charts = []

# Chart 1: Multi-company profit margin comparison
if has_multi and has_profit_margin:
    charts.append(('multi_profit', 'Profit Margin by Company'))
elif has_profit_margin and not has_multi:
    charts.append(('single_profit', 'Profit Margin Distribution'))

# Chart 2: ROA or efficiency analysis
if has_multi and has_roa:
    charts.append(('multi_roa', 'Return on Assets (ROA) by Company'))
elif has_roa and not has_multi:
    charts.append(('single_roa', 'Return on Assets Distribution'))
elif has_debt and not has_roa:
    charts.append(('single_debt', 'Debt Ratio Distribution'))

# Chart 3: Loss/profit distribution
if has_loss:
    charts.append(('loss_pie', 'Profit vs Loss Distribution'))

# Chart 4: Debt analysis (if not already used)
if has_debt and has_multi:
    charts.append(('multi_debt', 'Debt Ratio by Company'))

# Chart 5: Health score
if has_health:
    charts.append(('health_score', 'Financial Health Score Distribution'))

# Chart 6: Category analysis (Kaggle retail)
if has_category:
    charts.append(('category_profit', 'Profit by Category'))

# Chart 7: Region analysis (Kaggle retail)
if has_region and len(charts) < 6:
    charts.append(('region_profit', 'Profit by Region'))

# Chart 8: Discount analysis (Kaggle retail)
if has_discount and len(charts) < 6:
    charts.append(('discount_scatter', 'Discount vs Profit'))

# Chart 9: Stock returns
if has_stock:
    charts.append(('stock_returns', 'Daily Returns Distribution'))
    if len(charts) < 6:
        charts.append(('cumulative_returns', 'Cumulative Returns'))

# Chart 10: Trend over years
if has_year and has_profit_margin and len(charts) < 6:
    charts.append(('trend', 'Profit Margin Trend Over Years'))

# Ensure at least 2 charts
if len(charts) < 2:
    print("\n⚠️ Not enough data for charts. Showing basic data info.")
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.text(0.5, 0.5, 'Insufficient data for visualization\n\nPlease check your data contains:\n- Sales/Revenue and Profit columns\n- Or ticker column for multi-company\n- Or returns for stock analysis', 
            ha='center', va='center', transform=ax.transAxes, fontsize=12)
    ax.set_title(f'{COMPANY_NAME} - No Charts Available', fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    charts_filename = f"{clean_name}_charts.png"
    plt.savefig(charts_filename, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n✅ Chart saved as: {charts_filename}")
    print("   ℹ️ Only 1 chart generated due to insufficient data")
else:
    # Determine layout based on number of charts
    n_charts = min(len(charts), 6)
    if n_charts <= 2:
        fig, axes = plt.subplots(1, n_charts, figsize=(7 * n_charts, 5))
        axes = [axes] if n_charts == 1 else axes
    elif n_charts <= 4:
        fig, axes = plt.subplots(2, 2, figsize=(14, 12))
        axes = axes.flatten()
    else:
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        axes = axes.flatten()
    
    chart_count = 0
    
    for i, (chart_type, title) in enumerate(charts[:6]):
        ax = axes[i]
        
        if chart_type == 'multi_profit':
            company_margins = df.groupby('ticker')['profit_margin'].mean().sort_values(ascending=False)
            colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in company_margins.values]
            ax.bar(range(len(company_margins)), company_margins.values, color=colors)
            ax.set_xticks(range(len(company_margins)))
            ax.set_xticklabels(company_margins.index, rotation=45, ha='right')
            ax.set_ylabel('Profit Margin (%)')
            ax.set_title(title, fontsize=12, fontweight='bold')
            ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
        
        elif chart_type == 'single_profit':
            ax.hist(df['profit_margin'].dropna(), bins=25, color='#3498db', alpha=0.7, edgecolor='black')
            ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Break-even')
            ax.axvline(x=df['profit_margin'].mean(), color='green', linestyle='--', 
                      linewidth=2, label=f"Mean: {df['profit_margin'].mean():.1f}%")
            ax.set_xlabel('Profit Margin (%)')
            ax.set_ylabel('Frequency')
            ax.set_title(title, fontsize=12, fontweight='bold')
            ax.legend()
        
        elif chart_type == 'multi_roa':
            company_roa = df.groupby('ticker')['roa'].mean().sort_values(ascending=False)
            ax.bar(range(len(company_roa)), company_roa.values, color='#9b59b6')
            ax.set_xticks(range(len(company_roa)))
            ax.set_xticklabels(company_roa.index, rotation=45, ha='right')
            ax.set_ylabel('ROA (%)')
            ax.set_title(title, fontsize=12, fontweight='bold')
        
        elif chart_type == 'single_roa':
            ax.hist(df['roa'].dropna(), bins=25, color='#9b59b6', alpha=0.7, edgecolor='black')
            ax.axvline(x=df['roa'].mean(), color='red', linestyle='--', 
                      linewidth=2, label=f"Mean: {df['roa'].mean():.1f}%")
            ax.set_xlabel('ROA (%)')
            ax.set_ylabel('Frequency')
            ax.set_title(title, fontsize=12, fontweight='bold')
            ax.legend()
        
        elif chart_type == 'single_debt':
            ax.hist(df['debt_ratio'].dropna(), bins=25, color='#e74c3c', alpha=0.7, edgecolor='black')
            ax.axvline(x=df['debt_ratio'].mean(), color='blue', linestyle='--', 
                      linewidth=2, label=f"Mean: {df['debt_ratio'].mean():.1f}%")
            ax.axvline(x=50, color='red', linestyle='--', linewidth=1, label='Warning: 50%')
            ax.set_xlabel('Debt Ratio (%)')
            ax.set_ylabel('Frequency')
            ax.set_title(title, fontsize=12, fontweight='bold')
            ax.legend()
        
        elif chart_type == 'multi_debt':
            company_debt = df.groupby('ticker')['debt_ratio'].mean().sort_values(ascending=False)
            ax.bar(range(len(company_debt)), company_debt.values, color='#e74c3c')
            ax.set_xticks(range(len(company_debt)))
            ax.set_xticklabels(company_debt.index, rotation=45, ha='right')
            ax.set_ylabel('Debt Ratio (%)')
            ax.set_title(title, fontsize=12, fontweight='bold')
            ax.axhline(y=50, color='red', linestyle='--', linewidth=1)
        
        elif chart_type == 'loss_pie':
            loss_counts = df['is_loss'].value_counts()
            labels = ['Profitable', 'Loss-Making']
            sizes = [loss_counts.get(False, 0), loss_counts.get(True, 0)]
            colors_pie = ['#2ecc71', '#e74c3c']
            explode = (0, 0.05)
            ax.pie(sizes, labels=labels, colors=colors_pie, autopct='%1.1f%%', 
                   startangle=90, explode=explode, shadow=True)
            ax.set_title(title, fontsize=12, fontweight='bold')
        
        elif chart_type == 'health_score':
            score_col = 'financial_health_score' if 'financial_health_score' in df.columns else 'health_score'
            ax.hist(df[score_col].dropna(), bins=20, color='#1abc9c', alpha=0.7, edgecolor='black')
            ax.axvline(x=df[score_col].mean(), color='red', linestyle='--', 
                      linewidth=2, label=f"Mean: {df[score_col].mean():.1f}")
            ax.set_xlabel('Health Score (0-100)')
            ax.set_ylabel('Frequency')
            ax.set_title(title, fontsize=12, fontweight='bold')
            ax.legend()
        
        elif chart_type == 'category_profit':
            category_profit = df.groupby('category')['profit'].sum().sort_values(ascending=False)
            colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in category_profit.values]
            ax.bar(range(len(category_profit)), category_profit.values, color=colors)
            ax.set_xticks(range(len(category_profit)))
            ax.set_xticklabels(category_profit.index, rotation=45, ha='right')
            ax.set_ylabel('Profit ($)')
            ax.set_title(title, fontsize=12, fontweight='bold')
            ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
        
        elif chart_type == 'region_profit':
            region_profit = df.groupby('region')['profit'].sum().sort_values(ascending=False)
            ax.bar(range(len(region_profit)), region_profit.values, color='#3498db')
            ax.set_xticks(range(len(region_profit)))
            ax.set_xticklabels(region_profit.index, rotation=45, ha='right')
            ax.set_ylabel('Profit ($)')
            ax.set_title(title, fontsize=12, fontweight='bold')
            ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
        
        elif chart_type == 'discount_scatter':
            sample = df.sample(min(1000, len(df)))
            ax.scatter(sample['discount'], sample['profit'], alpha=0.4, c='#e74c3c')
            ax.axhline(y=0, color='blue', linestyle='--', linewidth=1)
            ax.set_xlabel('Discount Rate')
            ax.set_ylabel('Profit ($)')
            ax.set_title(title, fontsize=12, fontweight='bold')
        
        elif chart_type == 'stock_returns':
            ax.hist(df['ret'].dropna() * 100, bins=50, color='#3498db', alpha=0.7, edgecolor='black')
            ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero return')
            ax.axvline(x=df['ret'].mean() * 100, color='green', linestyle='--', 
                      linewidth=2, label=f"Mean: {df['ret'].mean() * 100:.2f}%")
            ax.set_xlabel('Daily Return (%)')
            ax.set_ylabel('Frequency')
            ax.set_title(title, fontsize=12, fontweight='bold')
            ax.legend()
        
        elif chart_type == 'cumulative_returns':
            df['cumulative'] = (1 + df['ret'].fillna(0)).cumprod() - 1
            date_col = 'date' if 'date' in df.columns else 'datadate' if 'datadate' in df.columns else None
            if date_col:
                ax.plot(df[date_col], df['cumulative'] * 100, color='#2ecc71', linewidth=2)
                ax.set_xlabel('Date')
            else:
                ax.plot(range(len(df)), df['cumulative'] * 100, color='#2ecc71', linewidth=2)
                ax.set_xlabel('Time')
            ax.set_ylabel('Cumulative Return (%)')
            ax.set_title(title, fontsize=12, fontweight='bold')
            ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
        
        elif chart_type == 'trend':
            if has_multi:
                for ticker in df['ticker'].unique():
                    ticker_data = df[df['ticker'] == ticker].sort_values('year')
                    ax.plot(ticker_data['year'], ticker_data['profit_margin'], 
                           marker='o', linewidth=2, label=ticker)
                ax.legend()
            else:
                yearly_data = df.groupby('year')['profit_margin'].mean()
                ax.plot(yearly_data.index, yearly_data.values, marker='o', linewidth=2, color='#3498db')
                ax.fill_between(yearly_data.index, yearly_data.values, alpha=0.3, color='#3498db')
            ax.set_xlabel('Year')
            ax.set_ylabel('Profit Margin (%)')
            ax.set_title(title, fontsize=12, fontweight='bold')
            ax.axhline(y=0, color='red', linestyle='--', linewidth=1)
            ax.grid(True, alpha=0.3)
        
        chart_count += 1
    
    # Hide unused subplots
    for i in range(chart_count, len(axes)):
        axes[i].set_visible(False)
    
    plt.tight_layout()
    charts_filename = f"{clean_name}_charts.png"
    plt.savefig(charts_filename, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n✅ Generated {chart_count} charts")
    print(f"✅ Charts saved as: {charts_filename}")

# =========================================================
# SAVE FILENAME FOR CELL 11 (THIS IS THE ONLY ADDITION)
# =========================================================
SAVED_CHARTS_FILENAME = charts_filename

print("\n" + "=" * 80)
print("                    VISUALIZATION COMPLETE!")
print("=" * 80)

## 11. Conclusions & Recommendations

**Key Findings** (auto-generated based on data):

- Profitability assessment
- Efficiency analysis (ROA/ROE)
- Leverage/Risk analysis (debt ratio)
- Loss analysis (loss rate)
- Financial health score (if available)
- Multi-company ranking (if multiple companies)

**Recommendations**:

- Specific recommendations based on findings
- General recommendations (monitoring, benchmarking, strategy)

In [ ]:
# ============================================
# UNIVERSAL CONCLUSIONS & RECOMMENDATIONS
# Auto-detects data and generates appropriate insights
# Works with: Multi/Single Company, Financial, Stock, Retail
# ============================================

print("\n" + "=" * 80)
print(f"                    {COMPANY_NAME}")
print("                    CONCLUSIONS & RECOMMENDATIONS")
print("=" * 80)

# =========================================================
# DETECT AVAILABLE DATA
# =========================================================
has_multi = 'ticker' in df.columns and len(df['ticker'].unique()) > 1
has_stock = 'ret' in df.columns
has_profit_margin = 'profit_margin' in df.columns
has_roa = 'roa' in df.columns
has_roe = 'roe' in df.columns
has_debt = 'debt_ratio' in df.columns
has_loss = 'is_loss' in df.columns
has_health = 'financial_health_score' in df.columns or 'health_score' in df.columns
has_category = 'category' in df.columns and 'profit' in df.columns
has_region = 'region' in df.columns and 'profit' in df.columns
has_year = 'year' in df.columns

print(f"\n📊 Data Overview:")
print(f"   Total records: {len(df):,} rows")
print(f"   Total columns: {len(df.columns)}")
if has_multi:
    print(f"   Companies: {', '.join(df['ticker'].unique())}")
if has_year:
    print(f"   Time period: {df['year'].min()} - {df['year'].max()}")

print("\n" + "=" * 80)
print("📊 KEY FINDINGS")
print("=" * 80)

finding_count = 1

# =========================================================
# PROFITABILITY ANALYSIS
# =========================================================
if has_profit_margin:
    avg_margin = df['profit_margin'].mean()
    print(f"\n{finding_count}. Profitability Analysis:")
    print(f"   - Average profit margin: {avg_margin:.2f}%")
    
    if has_multi:
        company_margins = df.groupby('ticker')['profit_margin'].mean().sort_values(ascending=False)
        best = company_margins.index[0]
        worst = company_margins.index[-1]
        print(f"   - Highest margin: {best} ({company_margins[best]:.2f}%)")
        print(f"   - Lowest margin: {worst} ({company_margins[worst]:.2f}%)")
        
        if company_margins[best] - company_margins[worst] > 10:
            print(f"   - ⚠️ Significant gap ({company_margins[best] - company_margins[worst]:.1f}%) between best and worst")
    
    if avg_margin > 20:
        print(f"   - ✅ Strong profitability - above industry average")
    elif avg_margin > 10:
        print(f"   - 📊 Moderate profitability - room for improvement")
    elif avg_margin > 0:
        print(f"   - ⚠️ Low profitability - needs attention")
    else:
        print(f"   - ❌ Negative profitability - urgent action required")
    
    finding_count += 1

# =========================================================
# EFFICIENCY ANALYSIS (ROA/ROE)
# =========================================================
if has_roa:
    avg_roa = df['roa'].mean()
    print(f"\n{finding_count}. Asset Efficiency (ROA):")
    print(f"   - Average ROA: {avg_roa:.2f}%")
    
    if has_multi:
        company_roa = df.groupby('ticker')['roa'].mean().sort_values(ascending=False)
        best = company_roa.index[0]
        print(f"   - Best performer: {best} ({company_roa[best]:.2f}%)")
    
    if avg_roa > 15:
        print(f"   - ✅ Excellent asset utilization")
    elif avg_roa > 10:
        print(f"   - 📊 Good asset efficiency")
    elif avg_roa > 5:
        print(f"   - ⚠️ Average efficiency - room for improvement")
    else:
        print(f"   - ❌ Poor asset efficiency")
    
    finding_count += 1

if has_roe:
    avg_roe = df['roe'].mean()
    print(f"\n{finding_count}. Shareholder Returns (ROE):")
    print(f"   - Average ROE: {avg_roe:.2f}%")
    
    if avg_roe > 20:
        print(f"   - ✅ Excellent shareholder returns")
    elif avg_roe > 15:
        print(f"   - 📊 Good returns")
    else:
        print(f"   - ⚠️ Below average returns")
    
    finding_count += 1

# =========================================================
# LEVERAGE/RISK ANALYSIS
# =========================================================
if has_debt:
    avg_debt = df['debt_ratio'].mean()
    print(f"\n{finding_count}. Financial Leverage:")
    print(f"   - Average debt/assets ratio: {avg_debt:.2f}%")
    
    if has_multi:
        company_debt = df.groupby('ticker')['debt_ratio'].mean().sort_values(ascending=False)
        highest = company_debt.index[0]
        print(f"   - Highest leverage: {highest} ({company_debt[highest]:.2f}%)")
    
    if avg_debt > 50:
        print(f"   - ⚠️ High leverage - refinancing risk")
        print(f"   - → Consider reducing debt levels")
    elif avg_debt > 30:
        print(f"   - 📊 Moderate leverage - acceptable")
    else:
        print(f"   - ✅ Low leverage - conservative financing")
        print(f"   - → Could consider strategic debt for growth")
    
    finding_count += 1

# =========================================================
# LOSS ANALYSIS
# =========================================================
if has_loss:
    loss_count = df['is_loss'].sum()
    loss_rate = (loss_count / len(df)) * 100
    print(f"\n{finding_count}. Loss Analysis:")
    print(f"   - Loss-making records: {loss_count} out of {len(df)} ({loss_rate:.2f}%)")
    
    if loss_rate > 30:
        print(f"   - ❌ High loss rate - significant profitability issues")
        print(f"   - → Identify and address loss-making segments urgently")
    elif loss_rate > 15:
        print(f"   - ⚠️ Moderate loss rate - review underperforming areas")
        print(f"   - → Conduct root cause analysis on loss-making items")
    elif loss_rate > 0:
        print(f"   - 📊 Low loss rate - mostly profitable")
        print(f"   - → Continue monitoring and optimize marginal cases")
    else:
        print(f"   - ✅ No loss-making records - excellent performance")
    
    finding_count += 1

# =========================================================
# FINANCIAL HEALTH SCORE
# =========================================================
if has_health:
    score_col = 'financial_health_score' if 'financial_health_score' in df.columns else 'health_score'
    avg_score = df[score_col].mean()
    print(f"\n{finding_count}. Financial Health Score:")
    print(f"   - Average score: {avg_score:.1f}/100")
    
    if avg_score >= 80:
        print(f"   - ✅ Excellent financial health - well-positioned for growth")
    elif avg_score >= 65:
        print(f"   - 📊 Good financial health - minor improvements possible")
    elif avg_score >= 50:
        print(f"   - ⚠️ Fair health - needs attention in key areas")
    else:
        print(f"   - ❌ Poor health - immediate action required")
    
    finding_count += 1

# =========================================================
# CATEGORY/RETAIL ANALYSIS
# =========================================================
if has_category:
    category_profit = df.groupby('category')['profit'].sum().sort_values(ascending=False)
    print(f"\n{finding_count}. Product Category Analysis:")
    for cat, profit in category_profit.items():
        status = "✅" if profit > 0 else "⚠️"
        print(f"   - {status} {cat}: ${profit:,.2f}")
    
    best_cat = category_profit.index[0]
    print(f"   - Best performing category: {best_cat}")
    
    if category_profit.iloc[-1] < 0:
        worst_cat = category_profit.index[-1]
        print(f"   - Loss-making category: {worst_cat} - needs review")
    
    finding_count += 1

# =========================================================
# REGION ANALYSIS
# =========================================================
if has_region:
    region_profit = df.groupby('region')['profit'].sum().sort_values(ascending=False)
    print(f"\n{finding_count}. Regional Analysis:")
    for reg, profit in region_profit.items():
        status = "✅" if profit > 0 else "⚠️"
        print(f"   - {status} {reg}: ${profit:,.2f}")
    
    best_region = region_profit.index[0]
    print(f"   - Best performing region: {best_region}")
    finding_count += 1

# =========================================================
# STOCK MARKET ANALYSIS
# =========================================================
if has_stock:
    avg_return = df['ret'].mean() * 100
    std_return = df['ret'].std() * 100
    total_return = ((1 + df['ret'].fillna(0)).prod() - 1) * 100
    
    print(f"\n{finding_count}. Stock Performance:")
    print(f"   - Average daily return: {avg_return:.4f}%")
    print(f"   - Volatility (daily std): {std_return:.4f}%")
    print(f"   - Total cumulative return: {total_return:.2f}%")
    
    if avg_return > 0:
        print(f"   - ✅ Positive average return")
    else:
        print(f"   - ⚠️ Negative average return")
    
    if std_return > 2:
        print(f"   - ⚠️ High volatility - higher risk")
    elif std_return > 1:
        print(f"   - 📊 Moderate volatility")
    else:
        print(f"   - ✅ Low volatility - stable")
    
    finding_count += 1

# =========================================================
# TREND ANALYSIS
# =========================================================
if has_year and has_profit_margin and len(df) > 5:
    yearly_data = df.groupby('year')['profit_margin'].mean()
    if len(yearly_data) > 1:
        first_val = yearly_data.iloc[0]
        last_val = yearly_data.iloc[-1]
        change = last_val - first_val
        
        print(f"\n{finding_count}. Trend Analysis:")
        print(f"   - Profit margin change: {change:+.2f}%")
        
        if change > 5:
            print(f"   - ✅ Strong upward trend - excellent improvement")
        elif change > 0:
            print(f"   - 📊 Positive trend - moving in right direction")
        elif change > -5:
            print(f"   - ⚠️ Slight decline - investigate causes")
        else:
            print(f"   - ❌ Significant decline - urgent action needed")
        
        finding_count += 1

# =========================================================
# RECOMMENDATIONS
# =========================================================
print("\n" + "=" * 80)
print("💡 RECOMMENDATIONS")
print("=" * 80)

rec_count = 1

# Specific recommendations based on findings
if has_profit_margin and df['profit_margin'].mean() < 15:
    print(f"\n   {rec_count}. 【Profitability】Review cost structure and optimize pricing")
    rec_count += 1

if has_multi and has_profit_margin:
    worst_company = df.groupby('ticker')['profit_margin'].mean().idxmin()
    print(f"\n   {rec_count}. 【Peer Analysis】Investigate why {worst_company} has lower margins than competitors")
    rec_count += 1

if has_roa and df['roa'].mean() < 10:
    print(f"\n   {rec_count}. 【Efficiency】Improve asset utilization and turnover")
    rec_count += 1

if has_debt:
    avg_debt = df['debt_ratio'].mean()
    if avg_debt > 50:
        print(f"\n   {rec_count}. 【Risk】Reduce debt levels to lower financial risk")
        rec_count += 1
    elif avg_debt < 20:
        print(f"\n   {rec_count}. 【Growth】Consider strategic debt for expansion opportunities")
        rec_count += 1

if has_loss and (df['is_loss'].sum() / len(df)) > 15:
    print(f"\n   {rec_count}. 【Loss Mitigation】Identify root causes of loss-making activities")
    rec_count += 1

if has_category and df.groupby('category')['profit'].sum().min() < 0:
    print(f"\n   {rec_count}. 【Product Strategy】Re-evaluate loss-making product categories")
    rec_count += 1

if has_region and df.groupby('region')['profit'].sum().min() < 0:
    print(f"\n   {rec_count}. 【Regional Strategy】Conduct cost audit for low-profit regions")
    rec_count += 1

if has_stock and df['ret'].std() * 100 > 2:
    print(f"\n   {rec_count}. 【Risk Management】Diversify portfolio to reduce volatility")
    rec_count += 1

# General recommendations (always shown)
print(f"\n   {rec_count}. 【Monitoring】Regularly track key financial ratios")
rec_count += 1
print(f"\n   {rec_count}. 【Benchmarking】Compare performance against industry peers")
rec_count += 1
print(f"\n   {rec_count}. 【Strategy】Align financial strategy with business goals")
rec_count += 1

print("\n" + "=" * 80)
print("                    ANALYSIS COMPLETE!")
print("=" * 80)

## 12. Export HTML Report

**Generated report includes**:

- Report summary (date, data source, record count)
- Key metrics cards (profit margin, ROA, debt ratio, health score)
- Multi-company comparison table (if applicable)
- Embedded charts from Cell 9
- Key findings (textual)
- Actionable recommendations

**Output file**: `{company_name}_complete_report.html`

In [ ]:
# ============================================
# FLEXIBLE HTML REPORT GENERATOR
# Reads data from df (already processed by Cells 6-8)
# Uses chart filename from Cell 9
# ============================================

print("\n" + "=" * 80)
print("                    GENERATING HTML REPORT")
print("=" * 80)

# Clean filename for saving
clean_name = COMPANY_NAME.replace(":", "").replace("/", "_").replace(" ", "_").replace(",", "")
clean_name = clean_name.replace("(", "").replace(")", "").replace("&", "and")

html_filename = f"{clean_name}_complete_report.html"

# =========================================================
# GET CHART FILENAME FROM CELL 9
# =========================================================
import base64

# Try to get the filename saved by Cell 9
try:
    charts_filename = SAVED_CHARTS_FILENAME
    with open(charts_filename, 'rb') as img_file:
        img_base64 = base64.b64encode(img_file.read()).decode('utf-8')
    chart_available = True
    print(f"   📊 Chart loaded: {charts_filename}")
except NameError:
    print("   ⚠️ Cell 9 hasn't been run or didn't save chart filename")
    chart_available = False
except FileNotFoundError:
    print(f"   ⚠️ Chart file not found: {charts_filename}")
    chart_available = False

# =========================================================
# AUTO-DETECT DATA TYPE (read from df, no hardcoding)
# =========================================================
has_multi = 'ticker' in df.columns and len(df['ticker'].unique()) > 1
has_stock = 'ret' in df.columns
has_retail_category = 'category' in df.columns and 'profit' in df.columns
has_retail_region = 'region' in df.columns and 'profit' in df.columns
has_discount = 'discount' in df.columns
has_health_score = 'financial_health_score' in df.columns or 'health_score' in df.columns
has_loss = 'is_loss' in df.columns
has_year = 'year' in df.columns

# Key metrics (already calculated in Cell 6-8)
metrics = {}
if 'profit_margin' in df.columns:
    metrics['profit_margin'] = df['profit_margin'].mean()
if 'roa' in df.columns:
    metrics['roa'] = df['roa'].mean()
if 'roe' in df.columns:
    metrics['roe'] = df['roe'].mean()
if 'debt_ratio' in df.columns:
    metrics['debt_ratio'] = df['debt_ratio'].mean()
if has_loss:
    metrics['loss_rate'] = (df['is_loss'].sum() / len(df)) * 100
if has_health_score:
    score_col = 'financial_health_score' if 'financial_health_score' in df.columns else 'health_score'
    metrics['health_score'] = df[score_col].mean()

# =========================================================
# BUILD HTML CONTENT
# =========================================================
html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{COMPANY_NAME} - Financial Analysis Report</title>
    <style>
        body {{
            font-family: 'Segoe UI', Arial, sans-serif;
            line-height: 1.6;
            color: #333;
            max-width: 1200px;
            margin: 0 auto;
            padding: 20px;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        }}
        .container {{ background-color: white; border-radius: 20px; padding: 30px; box-shadow: 0 20px 60px rgba(0,0,0,0.3); }}
        h1 {{ color: #2c3e50; border-bottom: 3px solid #3498db; padding-bottom: 10px; }}
        h2 {{ color: #34495e; border-bottom: 2px solid #ecf0f1; padding-bottom: 8px; margin-top: 30px; }}
        h3 {{ color: #555; margin-top: 20px; }}
        .summary-box {{ background: #f0f4f8; border-radius: 15px; padding: 20px; margin: 20px 0; }}
        .metric-container {{ display: flex; flex-wrap: wrap; gap: 15px; margin: 20px 0; }}
        .metric {{ flex: 1; min-width: 150px; padding: 15px; background: white; border-radius: 10px; text-align: center; box-shadow: 0 2px 5px rgba(0,0,0,0.1); }}
        .metric-value {{ font-size: 28px; font-weight: bold; color: #3498db; }}
        .metric-label {{ font-size: 12px; color: #666; margin-top: 5px; }}
        .chart-container {{ text-align: center; margin: 30px 0; background: #f8f9fa; padding: 20px; border-radius: 15px; }}
        .chart-container img {{ max-width: 100%; height: auto; border-radius: 10px; box-shadow: 0 4px 15px rgba(0,0,0,0.1); }}
        table {{ width: 100%; border-collapse: collapse; margin: 15px 0; }}
        th, td {{ border: 1px solid #ddd; padding: 10px; text-align: left; }}
        th {{ background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; }}
        .positive {{ color: #27ae60; font-weight: bold; }}
        .negative {{ color: #e74c3c; font-weight: bold; }}
        .footer {{ margin-top: 40px; padding-top: 20px; border-top: 1px solid #ddd; text-align: center; font-size: 12px; color: #999; }}
        ul {{ margin: 15px 0; padding-left: 25px; }}
        li {{ margin: 8px 0; }}
    </style>
</head>
<body>
    <div class="container">
        <h1>📊 {COMPANY_NAME}</h1>
        <h2>Financial Analysis Report</h2>
        
        <div class="summary-box">
            <h3>📋 Report Summary</h3>
            <p><strong>Analysis Date:</strong> {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
            <p><strong>Data Source:</strong> {DATA_SOURCE.upper()}</p>
            <p><strong>Records Analyzed:</strong> {len(df):,} rows, {len(df.columns)} columns</p>
"""

# Add company info
if has_multi:
    companies = df['ticker'].unique()
    html_content += f"<p><strong>Companies Analyzed:</strong> {', '.join(companies)}</p>"
if has_year:
    html_content += f"<p><strong>Time Period:</strong> {df['year'].min()} - {df['year'].max()}</p>"

html_content += """
        </div>
        
        <h2>📈 Key Metrics</h2>
        <div class="metric-container">
"""

# Add metrics dynamically
metric_labels = {
    'profit_margin': ('Avg Profit Margin', '%'),
    'roa': ('Avg Return on Assets', '%'),
    'roe': ('Avg Return on Equity', '%'),
    'debt_ratio': ('Avg Debt/Assets Ratio', '%'),
    'loss_rate': ('Loss Rate', '%'),
    'health_score': ('Financial Health Score', '/100')
}

for key, (label, unit) in metric_labels.items():
    if key in metrics:
        value = metrics[key]
        html_content += f"""
            <div class="metric">
                <div class="metric-value">{value:.1f}{unit}</div>
                <div class="metric-label">{label}</div>
            </div>
"""

html_content += """
        </div>
"""

# =========================================================
# MULTI-COMPANY COMPARISON TABLE (if applicable)
# =========================================================
if has_multi and 'profit_margin' in df.columns:
    html_content += """
        <h2>📊 Multi-Company Comparison</h2>
        <div class="summary-box">
        <table>
            <tr><th>Company</th><th>Profit Margin (%)</th><th>ROA (%)</th><th>Debt Ratio (%)</th></tr>
"""
    for ticker in df['ticker'].unique():
        ticker_data = df[df['ticker'] == ticker]
        margin = ticker_data['profit_margin'].mean() if 'profit_margin' in df.columns else 0
        roa = ticker_data['roa'].mean() if 'roa' in df.columns else 0
        debt = ticker_data['debt_ratio'].mean() if 'debt_ratio' in df.columns else 0
        html_content += f"<tr><td><strong>{ticker}</strong></td><td>{margin:.2f}%</td><td>{roa:.2f}%</td><td>{debt:.2f}%</td></tr>"
    html_content += "</table></div>"

# =========================================================
# CATEGORY ANALYSIS (if applicable)
# =========================================================
if has_retail_category:
    category_profit = df.groupby('category')['profit'].sum().sort_values(ascending=False)
    html_content += """
        <h2>📂 Product Category Analysis</h2>
        <div class="summary-box">
        <ul>
"""
    for cat, profit in category_profit.items():
        status = "✅" if profit > 0 else "⚠️"
        html_content += f"<li>{status} <strong>{cat}</strong>: ${profit:,.2f}</li>"
    html_content += "</ul></div>"

# =========================================================
# REGION ANALYSIS (if applicable)
# =========================================================
if has_retail_region:
    region_profit = df.groupby('region')['profit'].sum().sort_values(ascending=False)
    html_content += """
        <h2>📍 Regional Analysis</h2>
        <div class="summary-box">
        <ul>
"""
    for reg, profit in region_profit.items():
        status = "✅" if profit > 0 else "⚠️"
        html_content += f"<li>{status} <strong>{reg}</strong>: ${profit:,.2f}</li>"
    html_content += "</ul></div>"

# =========================================================
# CHARTS SECTION (embed from Cell 9)
# =========================================================
if chart_available:
    html_content += f"""
        <h2>📊 Analysis Charts</h2>
        <div class="chart-container">
            <img src="data:image/png;base64,{img_base64}" alt="Analysis Charts">
        </div>
"""

# =========================================================
# TEXTUAL FINDINGS (generated from data, not hardcoded)
# =========================================================
html_content += """
        <h2>🔍 Key Findings</h2>
        <div class="summary-box">
        <ul>
"""

# Generate findings based on available data
if 'profit_margin' in df.columns:
    avg_margin = df['profit_margin'].mean()
    if avg_margin > 20:
        html_content += f"<li>✅ <strong>Strong Profitability:</strong> Average profit margin is {avg_margin:.1f}%, above industry average.</li>"
    elif avg_margin > 10:
        html_content += f"<li>📊 <strong>Moderate Profitability:</strong> Average profit margin is {avg_margin:.1f}%, room for improvement.</li>"
    else:
        html_content += f"<li>⚠️ <strong>Low Profitability:</strong> Average profit margin is {avg_margin:.1f}%, needs attention.</li>"

if has_loss:
    loss_rate = (df['is_loss'].sum() / len(df)) * 100
    if loss_rate > 30:
        html_content += f"<li>⚠️ <strong>High Loss Rate:</strong> {loss_rate:.1f}% of records are loss-making.</li>"
    elif loss_rate > 15:
        html_content += f"<li>📊 <strong>Moderate Loss Rate:</strong> {loss_rate:.1f}% of records are loss-making.</li>"
    else:
        html_content += f"<li>✅ <strong>Low Loss Rate:</strong> Only {loss_rate:.1f}% of records are loss-making.</li>"

if 'roa' in df.columns:
    avg_roa = df['roa'].mean()
    if avg_roa > 15:
        html_content += f"<li>✅ <strong>Excellent Asset Efficiency:</strong> Average ROA is {avg_roa:.1f}%.</li>"
    elif avg_roa > 10:
        html_content += f"<li>📊 <strong>Good Asset Efficiency:</strong> Average ROA is {avg_roa:.1f}%.</li>"
    else:
        html_content += f"<li>⚠️ <strong>Room for Improvement:</strong> Average ROA is {avg_roa:.1f}%.</li>"

if 'debt_ratio' in df.columns:
    avg_debt = df['debt_ratio'].mean()
    if avg_debt > 50:
        html_content += f"<li>⚠️ <strong>High Leverage:</strong> Debt/Assets ratio is {avg_debt:.1f}%, refinancing risk.</li>"
    elif avg_debt > 30:
        html_content += f"<li>📊 <strong>Moderate Leverage:</strong> Debt/Assets ratio is {avg_debt:.1f}%, acceptable.</li>"
    else:
        html_content += f"<li>✅ <strong>Low Leverage:</strong> Debt/Assets ratio is {avg_debt:.1f}%, conservative financing.</li>"

if has_retail_category:
    best_cat = df.groupby('category')['profit'].sum().idxmax()
    html_content += f"<li>🏆 <strong>Top Performer:</strong> '{best_cat}' category generates the highest profit.</li>"

if has_retail_region:
    best_region = df.groupby('region')['profit'].sum().idxmax()
    html_content += f"<li>📍 <strong>Best Region:</strong> '{best_region}' region shows the strongest performance.</li>"

if has_multi and 'profit_margin' in df.columns:
    best_company = df.groupby('ticker')['profit_margin'].mean().idxmax()
    worst_company = df.groupby('ticker')['profit_margin'].mean().idxmin()
    html_content += f"<li>🏆 <strong>Margin Leader:</strong> {best_company} has the highest profit margin among peers.</li>"
    html_content += f"<li>📉 <strong>Margin Lagging:</strong> {worst_company} has the lowest profit margin, needs review.</li>"

if has_health_score:
    score_col = 'financial_health_score' if 'financial_health_score' in df.columns else 'health_score'
    avg_score = df[score_col].mean()
    if avg_score >= 80:
        html_content += f"<li>✅ <strong>Excellent Health:</strong> Financial health score is {avg_score:.0f}/100.</li>"
    elif avg_score >= 65:
        html_content += f"<li>📊 <strong>Good Health:</strong> Financial health score is {avg_score:.0f}/100.</li>"
    elif avg_score >= 50:
        html_content += f"<li>⚠️ <strong>Fair Health:</strong> Financial health score is {avg_score:.0f}/100.</li>"
    else:
        html_content += f"<li>❌ <strong>Poor Health:</strong> Financial health score is {avg_score:.0f}/100, immediate action needed.</li>"

html_content += """
        </ul>
        </div>
        
        <h2>💡 Recommendations</h2>
        <div class="summary-box">
        <ul>
"""

# Generate recommendations based on findings
rec_count = 1

if 'profit_margin' in df.columns and df['profit_margin'].mean() < 15:
    html_content += f"<li>{rec_count}. 【Profitability】Review cost structure and optimize pricing strategy.</li>"
    rec_count += 1

if has_multi and 'profit_margin' in df.columns:
    worst_company = df.groupby('ticker')['profit_margin'].mean().idxmin()
    best_company = df.groupby('ticker')['profit_margin'].mean().idxmax()
    html_content += f"<li>{rec_count}. 【Peer Analysis】Study {best_company}'s strategies to improve {worst_company}'s margins.</li>"
    rec_count += 1

if 'roa' in df.columns and df['roa'].mean() < 10:
    html_content += f"<li>{rec_count}. 【Efficiency】Improve asset utilization and turnover.</li>"
    rec_count += 1

if 'debt_ratio' in df.columns:
    avg_debt = df['debt_ratio'].mean()
    if avg_debt > 50:
        html_content += f"<li>{rec_count}. 【Risk】Reduce debt levels to lower financial risk.</li>"
        rec_count += 1
    elif avg_debt < 20:
        html_content += f"<li>{rec_count}. 【Growth】Consider strategic debt for expansion opportunities.</li>"
        rec_count += 1

if has_loss and (df['is_loss'].sum() / len(df)) > 15:
    html_content += f"<li>{rec_count}. 【Loss Mitigation】Identify root causes of loss-making activities.</li>"
    rec_count += 1

if has_retail_category:
    worst_cat = df.groupby('category')['profit'].sum().idxmin()
    if df.groupby('category')['profit'].sum()[worst_cat] < 0:
        html_content += f"<li>{rec_count}. 【Product Strategy】Re-evaluate '{worst_cat}' category - consider discontinuation or repricing.</li>"
        rec_count += 1

if has_retail_region:
    worst_region = df.groupby('region')['profit'].sum().idxmin()
    if df.groupby('region')['profit'].sum()[worst_region] < 0:
        html_content += f"<li>{rec_count}. 【Regional Strategy】Conduct cost audit for '{worst_region}' region.</li>"
        rec_count += 1

if has_discount and 'profit' in df.columns:
    high_discount = df[df['discount'] > 0.2]['profit'].mean() if len(df[df['discount'] > 0.2]) > 0 else 0
    low_discount = df[df['discount'] <= 0.2]['profit'].mean() if len(df[df['discount'] <= 0.2]) > 0 else 0
    if high_discount < low_discount:
        html_content += f"<li>{rec_count}. 【Discount Policy】Set discount caps (recommended ≤20%) to protect margins.</li>"
        rec_count += 1

# General recommendations
html_content += f"<li>{rec_count}. 【Monitoring】Regularly track key financial ratios quarterly.</li>"
rec_count += 1
html_content += f"<li>{rec_count}. 【Benchmarking】Compare performance against industry peers.</li>"
rec_count += 1
html_content += f"<li>{rec_count}. 【Strategy】Align financial strategy with business goals.</li>"

html_content += """
        </ul>
        </div>
        
        <div class="footer">
            <p>Generated by ACC102 Financial Analysis Tool</p>
            <p>This report is for educational purposes only.</p>
        </div>
    </div>
</body>
</html>
"""

# Write HTML file
with open(html_filename, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"\n✅ HTML report saved as: {html_filename}")
print(f"   📂 Location: {html_filename}")
print(f"   📊 Chart embedded: {'Yes' if chart_available else 'No'}")
print(f"   📈 Data type detected: {'Multi-company' if has_multi else 'Stock' if has_stock else 'Retail' if has_retail_category else 'Single-company'}")
print(f"\n   💡 To view: Double-click the file to open in your browser")

print("\n" + "=" * 80)
print("                    REPORT GENERATION COMPLETE!")
print("=" * 80)

## Summary

**Analysis Complete!**

The HTML report can be found in the same folder as this notebook. Double-click to open in your browser.

---

### Data Scope Note

This tool is designed for **corporate financial statement analysis** and requires datasets containing financial metrics (revenue/sales, profit/net income). It does not support non-financial data such as customer credit scores, loan risk assessment, or demographic data.

Please ensure your data meets these requirements before use.